# CP-1: road-arm movements for all four camera folders

This is a new experiment. The old notebooks, cached features and paper scores are retained.

1. Run all cells with `STAGE='calibrate'`. Download the generated `Draw_Camera_Gates.html` from Drive, open it in your browser, and draw each gate with the mouse. Click two endpoints (or drag a line), then click its junction side. Save each camera, download `gates_drawn.json`, then import it using the notebook's upload control. No coordinate sliders are needed.
2. Set `STAGE='pilot'`, run the configuration cell and the cells below it. Inspect the annotated pilot videos in Drive, then record your geometry/direction review with the displayed button. The default pilot covers all four cameras, including the three previously clear camera-2 clips.
3. Set `STAGE='batch'` and rerun configuration and the cells below it. Every MP4 under **normanniles1, normanniles2, normanniles3 and normanniles4** is included, including videos without congestion labels. There is no per-day sample limit. Extra folders such as `added` are outside these four views.
4. After completion, use `STAGE='evaluate'` to train/compare classifiers on clips with valid labels and three contiguous source clips of history. Default test dates were examined before; this remains exploratory.

All files are saved under **My Drive / CP-1 Dataset / Junction_v4**. Gates need actual road calibration; the notebook does not invent positions for unseen cameras. It requires annotated-pilot review for each camera before the full batch. A pilot is a geometry check, not a measurement of counting accuracy. Complete route counts require both gates within one clip; partial tracks are exported separately. Counts from overlapping camera views are not deduplicated across cameras.

In [1]:
# Cell 1 — install before importing the project modules. Use a Colab GPU runtime for inference.
import sys, subprocess, shutil, importlib.metadata
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'ultralytics==8.4.184','numpy==2.1.3','pandas==2.2.3','scikit-learn==1.6.1',
    'xgboost==3.4.1','opencv-python==4.11.0.86','ipywidgets','pytesseract','matplotlib'])
for name in ['numpy','pandas','sklearn']:
    installed=importlib.metadata.version('scikit-learn' if name=='sklearn' else name)
    if name in sys.modules and getattr(sys.modules[name],'__version__',installed)!=installed:
        raise RuntimeError('A loaded library was changed. Restart the Colab kernel and run from Cell 1.')
if shutil.which('tesseract') is None:
    try:
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','tesseract-ocr'],check=True)
    except (OSError,subprocess.CalledProcessError):
        print('Clock OCR unavailable. Counts can run, but camera-clock continuity will be marked unverified.')

In [2]:
# These embedded sources make this notebook independent of your local checkout.
from pathlib import Path
import sys, importlib
from google.colab import drive
drive.mount('/content/drive')
PROGRAM_FOLDER=Path('/content/junction_v4_code')
PROGRAM_FOLDER.mkdir(exist_ok=True)
MODULE_SOURCES = {'junction_counting_v4.py': '"""Camera-calibrated road-arm events and complete/incomplete vehicle journeys.\n\nNo road geometry is inferred from filenames or copied between cameras. Output\ncounts describe tracked journeys, not ground-truth vehicles. Clips are independent.\n"""\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport re\nimport shutil\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nVERSION = \'junction-v4.1\'\nCAMERAS = tuple(\'normanniles\'+str(i) for i in range(1, 5))\nCLASSES = {2: \'car\', 3: \'motorcycle\', 5: \'bus\', 7: \'truck\'}\nTRACKER = dict(tracker_type=\'bytetrack\', track_high_thresh=.4,\n               track_low_thresh=.05, new_track_thresh=.5,\n               track_buffer=90, match_thresh=.8, fuse_score=True)\nDEFAULTS = dict(imgsz=1280, conf=.05, anchor=\'bottom_center\',\n                max_gap_seconds=.5, max_motion_per_second=.8,\n                gate_margin=.002, clock_sample_seconds=1.,\n                clock_tolerance_seconds=2.5, save_observations=True)\nOBS_COLUMNS = [\'frame_idx\',\'track_id\',\'vehicle_class\',\'confidence\',\'x1\',\'y1\',\'x2\',\'y2\']\nEVENT_COLUMNS = [\'frame_idx\',\'track_key\',\'gate\',\'event\',\'vehicle_class\',\'confidence\',\'x\',\'y\']\nJOURNEY_COLUMNS = [\'track_key\',\'status\',\'origin\',\'destination\',\'reason\',\'vehicle_class\',\n                   \'first_frame\',\'last_frame\',\'entry_frame\',\'exit_frame\',\'mean_confidence\']\nQUALITY_COLUMNS = [\'frame_idx\',\'playback_seconds\',\'flag\',\'detail\']\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(\',\',\':\'),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    pending = path.with_name(path.name+\'.pending\')\n    pending.write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\')\n    pending.replace(path)\n\n\ndef write_csv(path, rows, columns):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        w = csv.DictWriter(f,fieldnames=columns,extrasaction=\'ignore\')\n        w.writeheader(); w.writerows(rows)\n\n\ndef cross(a,b,p):\n    return (b[0]-a[0])*(p[1]-a[1])-(b[1]-a[1])*(p[0]-a[0])\n\n\ndef intersection(a,b,p,q):\n    """Parameter of finite movement p->q intersecting finite gate a->b."""\n    r = (q[0]-p[0],q[1]-p[1]); s = (b[0]-a[0],b[1]-a[1])\n    den = r[0]*s[1]-r[1]*s[0]\n    if abs(den)<1e-12:\n        return None\n    ap = (a[0]-p[0],a[1]-p[1])\n    t = (ap[0]*s[1]-ap[1]*s[0])/den\n    u = (ap[0]*r[1]-ap[1]*r[0])/den\n    return t if 0<=t<=1 and 0<=u<=1 else None\n\n\ndef validate_camera(camera):\n    if not camera.get(\'reviewed\',False):\n        raise ValueError(\'Save reviewed gate positions for this camera first.\')\n    gates = camera.get(\'gates\',[])\n    if not 2<=len(gates)<=6:\n        raise ValueError(\'Define 2-6 road-arm gates (three for a three-arm junction).\')\n    names = [g[\'name\'] for g in gates]\n    if len(set(names))!=len(names) or any(not re.fullmatch(\'[A-F]\',n) for n in names):\n        raise ValueError(\'Use distinct gate IDs A-F; keep their road meanings fixed.\')\n    for gate in gates:\n        for key in [\'a\',\'b\',\'inside\']:\n            point = gate.get(key)\n            if not isinstance(point,(tuple,list)) or len(point)!=2 or any(\n                    not isinstance(x,(int,float)) or isinstance(x,bool)\n                    or not math.isfinite(x) or not 0<=x<=1 for x in point):\n                raise ValueError(f"{gate[\'name\']}: {key} must contain two normalized coordinates in [0,1].")\n        length = math.dist(gate[\'a\'],gate[\'b\'])\n        if length<.02 or abs(cross(gate[\'a\'],gate[\'b\'],gate[\'inside\']))/length<.005:\n            raise ValueError(\'Gate too short or its junction-side point is too close to the line.\')\n    for i,g in enumerate(gates):\n        for other in gates[i+1:]:\n            if intersection(g[\'a\'],g[\'b\'],other[\'a\'],other[\'b\']) is not None:\n                raise ValueError(\'Road-arm gates must not intersect one another.\')\n    size = camera.get(\'reference_size\')\n    if not size or len(size)!=2 or min(size)<=0:\n        raise ValueError(\'Missing reference frame size; run the reference-frame cell.\')\n    roi=camera.get(\'clock_roi\',[.68,0,1,.09])\n    if len(roi)!=4 or not (0<=roi[0]<roi[2]<=1 and 0<=roi[1]<roi[3]<=1):\n        raise ValueError(\'clock_roi must be [left, top, right, bottom] within [0,1].\')\n    return camera\n\n\ndef geometry(calibration, cameras=CAMERAS):\n    result = {}\n    for name in cameras:\n        if name not in calibration.get(\'cameras\',{}):\n            raise ValueError(\'Missing camera calibration: \'+name)\n        cam = validate_camera(calibration[\'cameras\'][name])\n        result[name] = {key:cam[key] for key in [\'gates\',\'reference_size\',\'reference_sha256\']}\n        result[name][\'clock_roi\']=cam.get(\'clock_roi\',[.68,0,1,.09])\n    return result\n\n\ndef inventory(base):\n    base = Path(base)\n    rows=[]\n    for camera in CAMERAS:\n        folder=base/camera\n        if not folder.is_dir():\n            raise FileNotFoundError(f\'Camera folder is inaccessible: {folder}\')\n        for p in sorted(folder.rglob(\'*\')):\n            if p.is_file() and p.suffix.lower()==\'.mp4\':\n                rows.append(dict(video_path=p.relative_to(base).as_posix(),\n                                 camera_id=camera,size_bytes=p.stat().st_size))\n    result=pd.DataFrame(rows,columns=[\'video_path\',\'camera_id\',\'size_bytes\'])\n    if result.empty or result.video_path.duplicated().any():\n        raise ValueError(\'No videos, or duplicate relative video paths.\')\n    missing=set(CAMERAS)-set(result.camera_id)\n    if missing:\n        raise ValueError(\'No videos found for \'+\', \'.join(sorted(missing)))\n    return result.sort_values(\'video_path\').reset_index(drop=True)\n\n\ndef reference_frames(base, videos, folder):\n    """Load one actual frame per camera without detector inference."""\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    target=folder/\'gates.json\'\n    calibration=json.loads(target.read_text()) if target.exists() else dict(version=VERSION,cameras={})\n    for camera in CAMERAS:\n        if camera in calibration[\'cameras\']:\n            if not (folder/(camera+\'.png\')).is_file():\n                raise FileNotFoundError(\'Existing calibration is missing its reference image: \'+camera)\n            if file_hash(folder/(camera+\'.png\'))!=calibration[\'cameras\'][camera][\'reference_sha256\']:\n                raise ValueError(\'Reference image changed; use a new calibration folder: \'+camera)\n            continue\n        choices=videos[videos.camera_id.eq(camera)].video_path.tolist()\n        preferred=\'normanniles2/normanniles2_2025-10-25-16-46-45.mp4\'\n        choices.sort(key=lambda k:(k!=preferred, not bool(re.search(r\'-(?:1[2-6])-\\d\\d-\\d\\d\\.mp4$\',k)),k))\n        found=False\n        for key in choices[:20]:\n            cap=cv2.VideoCapture(str(Path(base)/key))\n            try:\n                cap.set(cv2.CAP_PROP_POS_MSEC,1000)\n                ok,frame=cap.read()\n                if not ok: continue\n                h,w=frame.shape[:2]\n                image=folder/(camera+\'.png\')\n                if not cv2.imwrite(str(image),frame):raise OSError(\'Could not save \'+str(image))\n                calibration[\'cameras\'][camera]=dict(reviewed=False,gates=[],\n                    reference_video=key,reference_size=[w,h],reference_sha256=file_hash(image),\n                    clock_roi=[.68,0,1,.09])\n                found=True;break\n            finally:cap.release()\n        if not found:raise OSError(\'Could not decode a reference frame for \'+camera)\n    atomic_json(target,calibration)\n    return calibration\n\n\n@dataclass\nclass Track:\n    key: str\n    first: int\n    last: int\n    previous: tuple\n    anchors: dict = field(default_factory=dict)\n    votes: Counter = field(default_factory=Counter)\n    confidence: float = 0.\n    observations: int = 0\n    origin: str = \'\'\n    destination: str = \'\'\n    entry: object = None\n    exit: object = None\n    reason: str = \'\'\n    terminal: bool = False\n\n\nclass JourneyCounter:\n    """One observed journey per track segment; never guess missing gate events."""\n    def __init__(self,camera,fps,settings=None):\n        validate_camera(camera)\n        if not math.isfinite(fps) or fps<=0:raise ValueError(\'A valid frame rate is required.\')\n        self.gates=camera[\'gates\'];self.fps=fps\n        self.settings={**DEFAULTS,**(settings or {})}\n        self.tracks={};self.epochs=Counter();self.finished=[];self.events=[];self.issues=[]\n\n    def point(self,box,width,height):\n        return ((box[\'x1\']+box[\'x2\'])/(2*width),\n                (box[\'y2\'] if self.settings[\'anchor\']==\'bottom_center\' else (box[\'y1\']+box[\'y2\'])/2)/height)\n\n    def _side(self,gate,point):\n        dist=cross(gate[\'a\'],gate[\'b\'],point)/math.dist(gate[\'a\'],gate[\'b\'])\n        if abs(dist)<=self.settings[\'gate_margin\']:return 0\n        inside=1 if cross(gate[\'a\'],gate[\'b\'],gate[\'inside\'])>0 else -1\n        return (1 if dist>0 else -1)*inside\n\n    def observe(self,frame,track_id,point,vehicle_class,confidence):\n        if frame<0 or not math.isfinite(confidence) or not 0<=confidence<=1:\n            raise ValueError(\'Invalid frame/confidence observation.\')\n        if vehicle_class not in CLASSES.values():raise ValueError(\'Unknown vehicle class.\')\n        track_id=int(track_id); track=self.tracks.get(track_id)\n        if track is not None:\n            dt=(frame-track.last)/self.fps\n            if dt<=0:raise ValueError(\'Track observations must be unique and time-ordered.\')\n            reason=\'tracking_gap\' if dt>self.settings[\'max_gap_seconds\'] else (\n                \'position_jump\' if math.dist(point,track.previous)/dt>\n                self.settings[\'max_motion_per_second\'] else \'\')\n            if reason:\n                if not track.terminal:track.reason=reason\n                self.finished.append(track);del self.tracks[track_id];track=None\n                self.issues.append(dict(frame_idx=frame,flag=reason,detail=f\'track_id={track_id}\'))\n        if track is None:\n            self.epochs[track_id]+=1\n            track=Track(f\'{track_id}:{self.epochs[track_id]}\',frame,frame,point)\n            self.tracks[track_id]=track\n        track.votes[vehicle_class]+=1;track.confidence+=confidence;track.observations+=1\n        crossings=[]\n        for gate in self.gates:\n            name=gate[\'name\'];side=self._side(gate,point)\n            old=track.anchors.get(name)\n            if side and old is not None and old[0]!=side:\n                # Hysteresis anchors are checked against the actual finite segment.\n                # A track may dwell on the line while continuously observed.\n                # Actual observation gaps already split the track above.\n                t=intersection(gate[\'a\'],gate[\'b\'],old[1],point)\n                if t is not None:crossings.append((t,name,\'entry\' if side>0 else \'exit\'))\n            if side:track.anchors[name]=(side,point,frame)\n        if len(crossings)>1:\n            track.reason=\'ambiguous_multi_gate_step\';track.terminal=True\n            self.issues.append(dict(frame_idx=frame,flag=track.reason,detail=track.key))\n        elif crossings and not track.terminal:\n            _,gate,event=crossings[0]\n            self.events.append(dict(frame_idx=frame,track_key=track.key,gate=gate,event=event,\n                vehicle_class=vehicle_class,confidence=confidence,x=point[0],y=point[1]))\n            if event==\'entry\':\n                if track.origin:\n                    track.reason=\'multiple_entries_without_exit\';track.terminal=True\n                else:track.origin=gate;track.entry=frame\n            else:\n                track.destination=gate;track.exit=frame;track.terminal=True\n                track.reason=(\'complete\' if track.origin and track.origin!=gate else\n                              \'same_gate_return\' if track.origin else \'exit_without_entry\')\n        track.last=frame;track.previous=point\n\n    def expire(self,frame):\n        for key,track in list(self.tracks.items()):\n            if (frame-track.last)/self.fps>self.settings[\'max_gap_seconds\']:\n                if not track.terminal:track.reason=\'tracking_gap\' if track.origin else \'no_gate_observed\'\n                self.finished.append(track);del self.tracks[key]\n\n    def finish(self):\n        result=[]\n        for t in self.finished+list(self.tracks.values()):\n            reason=t.reason or (\'entry_without_exit\' if t.origin else \'no_gate_observed\')\n            cls=sorted(t.votes,key=lambda c:(-t.votes[c],c))[0]\n            result.append(dict(track_key=t.key,status=\'complete\' if reason==\'complete\' else \'incomplete\',\n                origin=t.origin or \'unknown\',destination=t.destination or \'unknown\',reason=reason,\n                vehicle_class=cls,first_frame=t.first,last_frame=t.last,entry_frame=t.entry,\n                exit_frame=t.exit,mean_confidence=t.confidence/t.observations))\n        return result\n\n\nclass FrameQuality:\n    """Candidate flags only: never relabel congestion, remove clips or fabricate frames."""\n    def __init__(self,fps,camera,settings):\n        self.fps=fps;self.camera=camera;self.settings=settings\n        self.previous=None;self.previous_clock=None;self.flags=[];self.ocr_reads=0\n        self.ocr_matches=0;self.ocr_error=\'\'\n        try:\n            import pytesseract\n            pytesseract.get_tesseract_version();self.ocr=pytesseract\n        except (ImportError,OSError,RuntimeError) as e:\n            self.ocr=None;self.ocr_error=str(e)[:200]\n\n    def add(self,frame,flag,detail):\n        self.flags.append(dict(frame_idx=frame,playback_seconds=round(frame/self.fps,3),flag=flag,detail=detail))\n\n    def check(self,index,frame):\n        interval=self.settings[\'clock_sample_seconds\']\n        if interval<=0 or index%max(1,round(self.fps*interval)):return\n        tiny=cv2.resize(frame,(80,45));gray=cv2.cvtColor(tiny,cv2.COLOR_BGR2GRAY)\n        lab=cv2.cvtColor(tiny,cv2.COLOR_BGR2LAB).astype(float)\n        if self.previous is not None:\n            oldgray,oldlab=self.previous\n            luma=float(abs(gray.mean()-oldgray.mean()))\n            chroma=float(np.linalg.norm(lab[:,:,1:].mean((0,1))-oldlab[:,:,1:].mean((0,1))))\n            if luma>25 or chroma>12:\n                self.add(index,\'colour_brightness_change_candidate\',f\'luma_delta={luma:.2f}; chroma_delta={chroma:.2f}\')\n        self.previous=(gray,lab)\n        if self.ocr is None:return\n        h,w=frame.shape[:2];x1,y1,x2,y2=self.camera.get(\'clock_roi\',[.68,0,1,.09])\n        crop=frame[round(y1*h):round(y2*h),round(x1*w):round(x2*w)]\n        if not crop.size:return\n        crop=cv2.cvtColor(cv2.resize(crop,None,fx=3,fy=3),cv2.COLOR_BGR2GRAY)\n        self.ocr_reads+=1\n        try:\n            text=self.ocr.image_to_string(crop,config=\'--psm 7\',timeout=3)\n        except (RuntimeError,OSError) as e:\n            self.ocr_error=str(e)[:200];return\n        matches=re.findall(r\'(\\d{1,2}):(\\d{2}):(\\d{2})\',text)\n        valid=[tuple(map(int,m)) for m in matches if int(m[0])<24 and int(m[1])<60 and int(m[2])<60]\n        if not valid:return\n        hh,mm,ss=valid[-1];seconds=hh*3600+mm*60+ss;self.ocr_matches+=1\n        if self.previous_clock is not None:\n            old,old_index=self.previous_clock\n            delta=seconds-old\n            if old>23*3600 and seconds<3600:delta+=86400\n            elapsed=(index-old_index)/self.fps\n            if abs(delta-elapsed)>self.settings[\'clock_tolerance_seconds\']:\n                self.add(index,\'camera_clock_jump_candidate\',\n                         f\'OCR clock_delta={delta}; playback_delta={elapsed:.3f}; raw={text.strip()}\')\n        self.previous_clock=(seconds,index)\n\n\ndef summarize(counter,journeys,frame_counts,areas,confidence,speeds,frames,fps,width,height):\n    complete=[j for j in journeys if j[\'status\']==\'complete\']\n    result=dict(total_frames=frames,fps=fps,duration_sec=frames/fps,\n        resolution=f\'{width}x{height}\',complete_journeys=len(complete),\n        incomplete_tracks=len(journeys)-len(complete),observed_tracks=len(journeys),\n        gate_entries=sum(e[\'event\']==\'entry\' for e in counter.events),\n        gate_exits=sum(e[\'event\']==\'exit\' for e in counter.events),\n        mean_vehicles=float(np.mean(frame_counts)),max_vehicles=int(max(frame_counts,default=0)),\n        density_std=float(np.std(frame_counts)),\n        density_trend=float(np.mean(frame_counts[-max(1,frames//3):])-np.mean(frame_counts[:max(1,frames//3)])),\n        mean_bbox_area_fraction=float(np.mean(areas)) if areas else 0.,\n        bbox_occupancy_proxy=float(sum(areas)/frames),\n        mean_confidence=float(np.mean(confidence)) if confidence else 0.,\n        motion_median=float(np.median(speeds)) if speeds else 0.,\n        motion_p90=float(np.quantile(speeds,.9)) if speeds else 0.,\n        slow_fraction=float(np.mean(np.array(speeds)<.005)) if speeds else 0.,\n        weighted_complete_journeys=float(sum(j[\'mean_confidence\'] for j in complete)))\n    for gate in sorted(g[\'name\'] for g in counter.gates):\n        for direction in [\'entry\',\'exit\']:\n            result[f\'gate_{gate}_{direction}\']=sum(e[\'gate\']==gate and e[\'event\']==direction for e in counter.events)\n            for cls in CLASSES.values():\n                result[f\'gate_{gate}_{direction}_{cls}\']=sum(e[\'gate\']==gate and e[\'event\']==direction\n                    and e[\'vehicle_class\']==cls for e in counter.events)\n    for a in sorted(g[\'name\'] for g in counter.gates):\n        for b in sorted(g[\'name\'] for g in counter.gates):\n            if a!=b:\n                movement=[j for j in complete if j[\'origin\']==a and j[\'destination\']==b]\n                result[f\'route_{a}_{b}\']=len(movement)\n                for cls in CLASSES.values():result[f\'route_{a}_{b}_{cls}\']=sum(j[\'vehicle_class\']==cls for j in movement)\n    for cls in CLASSES.values():\n        result[\'journeys_\'+cls]=sum(j[\'vehicle_class\']==cls for j in complete)\n    return result\n\n\ndef draw_gates(frame,camera):\n    out=frame.copy();h,w=frame.shape[:2]\n    for gate in camera.get(\'gates\',[]):\n        a=tuple(round(v*s) for v,s in zip(gate[\'a\'],(w,h)))\n        b=tuple(round(v*s) for v,s in zip(gate[\'b\'],(w,h)))\n        inside=tuple(round(v*s) for v,s in zip(gate[\'inside\'],(w,h)))\n        middle=tuple(round((x+y)/2) for x,y in zip(a,b))\n        cv2.line(out,a,b,(0,255,255),2);cv2.arrowedLine(out,middle,inside,(0,255,0),2)\n        cv2.putText(out,gate[\'name\']+\' \'+gate.get(\'label\',\'\'),a,cv2.FONT_HERSHEY_SIMPLEX,.55,(0,255,255),2)\n    return out\n\n\ndef reset_tracker(model):\n    for tracker in getattr(getattr(model,\'predictor\',None),\'trackers\',[]) or []:tracker.reset()\n\n\ndef process_clip(video,camera,model,output,tracker_path,settings=None,preview=False):\n    """Full decode only. Failed or interrupted clips cannot become checkpoints."""\n    settings={**DEFAULTS,**(settings or {})};validate_camera(camera)\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    cap=cv2.VideoCapture(str(video));writer=None\n    if not cap.isOpened():cap.release();raise OSError(\'Cannot open \'+str(video))\n    fps=float(cap.get(cv2.CAP_PROP_FPS));expected=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    try:\n        if min(width,height)<=0 or fps<=0 or not math.isfinite(fps):raise ValueError(\'Invalid video metadata.\')\n        rw,rh=camera[\'reference_size\']\n        if abs(width/height-rw/rh)>.01:raise ValueError(\'Aspect ratio differs from calibrated reference.\')\n        counter=JourneyCounter(camera,fps,settings);quality=FrameQuality(fps,camera,settings)\n        reset_tracker(model)\n        if preview:\n            writer=cv2.VideoWriter(str(output/\'annotated.mp4\'),cv2.VideoWriter_fourcc(*\'mp4v\'),fps,(width,height))\n            if not writer.isOpened():raise OSError(\'Annotated video writer did not open.\')\n        frames=0;densities=[];areas=[];scores=[];speeds=[];last={}\n        obs_file=(output/\'observations.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\')\n        with obs_file:\n            obs=csv.DictWriter(obs_file,fieldnames=OBS_COLUMNS);obs.writeheader()\n            while True:\n                ok,frame=cap.read()\n                if not ok:break\n                quality.check(frames,frame)\n                result=model.track(frame,persist=True,tracker=str(tracker_path),\n                    imgsz=settings[\'imgsz\'],conf=settings[\'conf\'],verbose=False,classes=list(CLASSES))[0]\n                count=0;annotated=draw_gates(frame,camera) if preview else None\n                boxes=getattr(result,\'boxes\',None)\n                if boxes is not None and boxes.id is not None:\n                    seen=set()\n                    for box in boxes:\n                        track_id=int(box.id[0]);cls=CLASSES.get(int(box.cls[0]))\n                        if cls is None:continue\n                        if track_id in seen:raise ValueError(\'Duplicate track ID in a frame.\')\n                        seen.add(track_id)\n                        x1,y1,x2,y2=map(float,box.xyxy[0].cpu().numpy());conf=float(box.conf[0])\n                        row=dict(frame_idx=frames,track_id=track_id,vehicle_class=cls,\n                                 confidence=conf,x1=x1,y1=y1,x2=x2,y2=y2)\n                        point=counter.point(row,width,height)\n                        if track_id in last:\n                            old_frame,old=last[track_id];dt=(frames-old_frame)/fps\n                            if 0<dt<=settings[\'max_gap_seconds\']:\n                                speed=math.dist(point,old)/dt\n                                if speed<=settings[\'max_motion_per_second\']:speeds.append(speed)\n                        last[track_id]=(frames,point)\n                        counter.observe(frames,track_id,point,cls,conf)\n                        if settings[\'save_observations\']:obs.writerow(row)\n                        count+=1;areas.append(max(0,x2-x1)*max(0,y2-y1)/(width*height));scores.append(conf)\n                        if preview:\n                            cv2.rectangle(annotated,(int(x1),int(y1)),(int(x2),int(y2)),(255,160,0),1)\n                            cv2.circle(annotated,(round(point[0]*width),round(point[1]*height)),3,(0,0,255),-1)\n                            cv2.putText(annotated,f\'{cls} #{track_id}\',(int(x1),max(15,int(y1)-5)),cv2.FONT_HERSHEY_SIMPLEX,.4,(255,160,0),1)\n                counter.expire(frames);densities.append(count)\n                if preview:writer.write(annotated)\n                frames+=1\n        if frames==0:raise OSError(\'No frames decoded.\')\n        if expected>0 and frames<expected:\n            raise OSError(f\'Incomplete decode: {frames}/{expected} frames. Clip not checkpointed.\')\n        journeys=counter.finish()\n        features=summarize(counter,journeys,densities,areas,scores,speeds,frames,fps,width,height)\n        for issue in counter.issues:quality.add(issue[\'frame_idx\'],issue[\'flag\'],issue[\'detail\'])\n        features.update(quality_flags=len(quality.flags),clock_ocr_reads=quality.ocr_reads,\n                        clock_ocr_matches=quality.ocr_matches,\n                        clock_check_status=\'sampled\' if quality.ocr_matches else \'not_verified\',\n                        quality_review_status=\'candidates_present\' if quality.flags else \'no_automatic_candidates\')\n        write_csv(output/\'gate_events.csv\',counter.events,EVENT_COLUMNS)\n        write_csv(output/\'journeys.csv\',journeys,JOURNEY_COLUMNS)\n        write_csv(output/\'quality_flags.csv\',quality.flags,QUALITY_COLUMNS)\n        write_csv(output/\'frame_counts.csv\',[dict(frame_idx=i,tracked_vehicles=n) for i,n in enumerate(densities)],\n                  [\'frame_idx\',\'tracked_vehicles\'])\n        atomic_json(output/\'features.json\',features)\n        return features,dict(expected_frames=expected,decoded_frames=frames,clock_ocr_error=quality.ocr_error)\n    finally:\n        cap.release()\n        if writer is not None:writer.release()\n\n\ndef clip_directory(run,key):\n    parts=Path(key).parts\n    if len(parts)<2 or parts[0] not in CAMERAS or \'..\' in parts or Path(key).is_absolute():\n        raise ValueError(\'Unsafe video key: \'+key)\n    return Path(run)/\'clips\'/parts[0]/(Path(key).stem+\'_\'+digest(key)[:10])\n\n\ndef verify_checkpoint(folder,signature,video=None,expected_weights=None):\n    folder=Path(folder);manifest=folder/\'manifest.json\'\n    if not manifest.is_file():raise ValueError(\'Incomplete checkpoint: \'+str(folder))\n    data=json.loads(manifest.read_text())\n    if data[\'contract_signature\']!=signature:raise ValueError(\'Checkpoint contract differs.\')\n    if expected_weights is not None and data.get(\'detector_weights\')!=expected_weights:\n        raise ValueError(\'Checkpoint detector weights differ from the run registry.\')\n    for name,expected in data[\'output_sha256\'].items():\n        if not (folder/name).is_file() or file_hash(folder/name)!=expected:\n            raise ValueError(\'Checkpoint file changed: \'+str(folder/name))\n    if video is not None and file_hash(video)!=data[\'source_sha256\']:\n        raise ValueError(\'Source video changed: \'+str(video))\n    return json.loads((folder/\'features.json\').read_text())\n\n\ndef freeze_run(run,videos,calibration,detector,settings=None):\n    run=Path(run);run.mkdir(parents=True,exist_ok=True)\n    if detector not in [\'yolo11\',\'yolov8\']:raise ValueError(\'Choose yolo11 or yolov8.\')\n    names=sorted(set(videos.camera_id))\n    config=dict(version=VERSION,code_sha256=file_hash(__file__),detector=detector,\n                settings={**DEFAULTS,**(settings or {})},tracker=TRACKER,\n                geometry=geometry(calibration,names),\n                inventory=videos[[\'video_path\',\'camera_id\',\'size_bytes\']].to_dict(\'records\'),\n                environment={n:importlib.metadata.version(n) for n in [\'ultralytics\',\'opencv-python\',\'numpy\',\'torch\']})\n    signature=digest(config);path=run/\'contract.json\'\n    if path.exists():\n        saved=json.loads(path.read_text())\n        if saved.get(\'signature\')!=digest(saved[\'config\']):raise ValueError(\'Saved contract is damaged.\')\n        if saved[\'signature\']!=signature:\n            changed=[k for k in config if config[k]!=saved[\'config\'].get(k)]\n            detail=(\' Saved environment: \'+str(saved[\'config\'].get(\'environment\'))+\n                    \'; current environment: \'+str(config[\'environment\'])) if \'environment\' in changed else \'\'\n            raise ValueError(\'Cannot resume; changed fields: \'+\', \'.join(changed)+\'.\'+detail+\n                \' Restore the original settings/packages to resume, or use a new EXPERIMENT_NAME for an intentional change. Saved clips are intact.\')\n    else:atomic_json(path,dict(signature=signature,config=config))\n    return signature\n\n\ndef collect_features(run,videos,signature,verify_files=True):\n    rows=[];missing=[]\n    registry=Path(run)/\'model_weights.json\'\n    expected_weights=json.loads(registry.read_text()) if registry.exists() else None\n    for r in videos.itertuples():\n        folder=clip_directory(run,r.video_path)\n        if not folder.exists():missing.append(r.video_path);continue\n        if expected_weights is None:raise ValueError(\'Missing detector-weights registry for saved checkpoints.\')\n        feat=verify_checkpoint(folder,signature,expected_weights=expected_weights) if verify_files else json.loads((folder/\'features.json\').read_text())\n        rows.append(dict(video_path=r.video_path,camera_id=r.camera_id,**feat))\n    result=pd.DataFrame(rows)\n    if not result.empty:\n        pending_file=Path(run)/\'junction_features.csv.pending\'\n        result.to_csv(pending_file,index=False)\n        pending_file.replace(Path(run)/\'junction_features.csv\')\n    write_csv(Path(run)/\'pending_videos.csv\',[dict(video_path=k) for k in missing],[\'video_path\'])\n    atomic_json(Path(run)/\'batch_status.json\',dict(planned=len(videos),completed=len(rows),\n        pending=len(missing),contract_signature=signature,all_videos_completed=not missing))\n    return result,missing\n\n\ndef run_batch(base,run,videos,calibration,detector=\'yolo11\',settings=None,preview=False,model_factory=None):\n    """Process all rows in the frozen inventory; restart interrupted clips at zero."""\n    import yaml\n    settings={**DEFAULTS,**(settings or {})}\n    signature=freeze_run(run,videos,calibration,detector,settings)\n    run=Path(run);tracker=run/\'bytetrack.yaml\'\n    tracker.write_text(yaml.safe_dump(TRACKER),encoding=\'utf-8\')\n    weights=\'yolo11l.pt\' if detector==\'yolo11\' else \'yolov8l.pt\'\n    registry=run/\'model_weights.json\'\n    weight_record=json.loads(registry.read_text()) if registry.exists() else None\n    failures=[];model=None;started=time.monotonic();completed=0;consecutive_failures=0\n    for index,row in enumerate(videos.itertuples(),1):\n        source=Path(base)/row.video_path;dest=clip_directory(run,row.video_path)\n        if dest.exists():\n            if weight_record is None:raise ValueError(\'Missing detector-weights registry for saved checkpoints.\')\n            verify_checkpoint(dest,signature,source,weight_record)\n            print(f\'[{index}/{len(videos)}] Already checkpointed: {row.video_path}\');continue\n        stage=run/\'.pending\'/digest(row.video_path)\n        if stage.exists():\n            if stage.resolve().parent!=(run/\'.pending\').resolve() or not re.fullmatch(\'[a-f0-9]{64}\',stage.name):\n                raise ValueError(\'Unsafe staging directory.\')\n            shutil.rmtree(stage)\n        stage.mkdir(parents=True,exist_ok=True)\n        try:\n            if model is None:\n                if model_factory is None:\n                    from ultralytics import YOLO\n                    model=YOLO(weights)\n                else:model=model_factory(weights)\n                weight_path=Path(getattr(model,\'ckpt_path\',weights))\n                if model_factory is None and not weight_path.is_file():\n                    raise FileNotFoundError(\'Cannot fingerprint detector weights: \'+str(weight_path))\n                weight_record=dict(filename=weights,sha256=file_hash(weight_path) if weight_path.is_file() else \'test_model_factory\')\n                registry=run/\'model_weights.json\'\n                if registry.exists() and json.loads(registry.read_text())!=weight_record:\n                    raise ValueError(\'Detector weights changed. Restore the original weights or use a new experiment.\')\n                atomic_json(registry,weight_record)\n            before_hash=file_hash(source)\n            clip_started=time.monotonic()\n            print(f\'[{index}/{len(videos)}] Processing: {row.video_path}\')\n            features,meta=process_clip(source,calibration[\'cameras\'][row.camera_id],model,\n                                      stage,tracker,settings,preview)\n            if file_hash(source)!=before_hash:raise ValueError(\'Source changed while processing.\')\n            hashes={p.name:file_hash(p) for p in stage.iterdir() if p.is_file()}\n            manifest=dict(video_path=row.video_path,model_weights=weights,\n                detector_weights=weight_record,\n                contract_signature=signature,source_sha256=before_hash,output_sha256=hashes,\n                processing_seconds=time.monotonic()-clip_started,\n                recorded_utc=datetime.now(timezone.utc).isoformat(),**meta)\n            atomic_json(stage/\'manifest.json\',manifest)\n            dest.parent.mkdir(parents=True,exist_ok=True);stage.replace(dest);completed+=1\n            consecutive_failures=0\n            elapsed=time.monotonic()-started\n            print(f"  Complete journeys: {features[\'complete_journeys\']}; incomplete tracks: {features[\'incomplete_tracks\']}; new-clip mean: {elapsed/completed/60:.2f} min")\n        except KeyboardInterrupt:\n            print(\'Stopped. Completed clips are saved; this unfinished clip will restart at frame zero.\')\n            raise\n        except Exception as error:\n            consecutive_failures+=1\n            failures.append(dict(video_path=row.video_path,error=type(error).__name__+\': \'+str(error)))\n            write_csv(run/\'processing_failures.csv\',failures,[\'video_path\',\'error\'])\n            print(\'  Failed; no completed checkpoint:\',error)\n            if consecutive_failures>=3:\n                raise RuntimeError(\'Three consecutive clips failed. Resolve the logged error, then resume this experiment.\') from error\n    write_csv(run/\'processing_failures.csv\',failures,[\'video_path\',\'error\'])\n    features,missing=collect_features(run,videos,signature)\n    atomic_json(run/\'batch_status.json\',dict(planned=len(videos),completed=len(features),\n        pending=len(missing),contract_signature=signature,all_videos_completed=not missing))\n    return features,missing\n', 'junction_calibration_v4.py': '"""Standard ipywidgets calibration controls, usable with VS Code/Colab notebooks."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport importlib.metadata\nimport json\nimport zipfile\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nimport junction_counting_v4 as core\n\n\ndef gate_editor(folder):\n    import ipywidgets as w\n    import matplotlib.pyplot as plt\n    from IPython.display import display,clear_output\n    folder=Path(folder);path=folder/\'gates.json\'\n    calibration=json.loads(path.read_text())\n    camera=w.Dropdown(options=core.CAMERAS,description=\'Camera:\')\n    gate=w.Dropdown(options=[\'A\',\'B\',\'C\'],description=\'Road gate:\')\n    label=w.Text(description=\'Road name:\',placeholder=\'e.g. left / upper / right\')\n    names=[\'a_x\',\'a_y\',\'b_x\',\'b_y\',\'inside_x\',\'inside_y\']\n    sliders={n:w.IntSlider(description=n,min=0,max=100,value=0,continuous_update=False,\n                           layout=w.Layout(width=\'430px\')) for n in names}\n    status=w.Output();plot=w.Output();draft={};loading=[False]\n    store=w.Button(description=\'Store this road gate\',button_style=\'info\',layout=w.Layout(width=\'210px\'))\n    save=w.Button(description=\'Save reviewed camera gates\',button_style=\'success\',layout=w.Layout(width=\'260px\'))\n\n    def candidate():\n        rw,rh=calibration[\'cameras\'][camera.value][\'reference_size\']\n        return dict(name=gate.value,label=label.value.strip(),\n            a=[sliders[\'a_x\'].value/rw,sliders[\'a_y\'].value/rh],\n            b=[sliders[\'b_x\'].value/rw,sliders[\'b_y\'].value/rh],\n            inside=[sliders[\'inside_x\'].value/rw,sliders[\'inside_y\'].value/rh])\n\n    def render(*args):\n        if loading[0]:return\n        image=cv2.imread(str(folder/(camera.value+\'.png\')))\n        if image is None:raise FileNotFoundError(\'Reference image is missing.\')\n        display_camera={**calibration[\'cameras\'][camera.value],\n                        \'gates\':[g for n,g in draft.items() if n!=gate.value]+[candidate()]}\n        shown=core.draw_gates(image,display_camera)\n        with plot:\n            clear_output(wait=True)\n            fig,ax=plt.subplots(figsize=(12,6.75))\n            ax.imshow(cv2.cvtColor(shown,cv2.COLOR_BGR2RGB))\n            h,width=image.shape[:2]\n            ax.set_xticks(range(0,width+1,max(1,width//10)));ax.set_yticks(range(0,h+1,max(1,h//10)))\n            ax.grid(alpha=.35);ax.set_title(camera.value+\' — yellow gate, green arrow toward junction\')\n            plt.show();plt.close(fig)\n\n    def load_gate(*args):\n        loading[0]=True\n        size=calibration[\'cameras\'][camera.value][\'reference_size\']\n        g=draft.get(gate.value,dict(label=\'\',a=[0,0],b=[0,0],inside=[0,0]))\n        label.value=g.get(\'label\',\'\')\n        vals=[*g[\'a\'],*g[\'b\'],*g[\'inside\']]\n        for i,(name,value) in enumerate(zip(names,vals)):\n            sliders[name].max=size[i%2];sliders[name].value=round(value*size[i%2])\n        loading[0]=False;render()\n\n    def load_camera(*args):\n        draft.clear();draft.update({g[\'name\']:g for g in calibration[\'cameras\'][camera.value][\'gates\']})\n        load_gate()\n\n    def store_gate(button):\n        g=candidate()\n        with status:\n            clear_output()\n            if core.math.dist(g[\'a\'],g[\'b\'])<.02:\n                print(\'Move the two endpoints so the gate spans the road.\');return\n            if abs(core.cross(g[\'a\'],g[\'b\'],g[\'inside\']))/core.math.dist(g[\'a\'],g[\'b\'])<.005:\n                print(\'Move the inside point onto the junction side of the gate.\');return\n            draft[gate.value]=g\n            print(\'Stored gate\',gate.value,\'for\',camera.value,\'in the editor. Store all road gates, then save this camera.\')\n        render()\n\n    def save_camera(button):\n        with status:\n            clear_output()\n            try:\n                if set(draft)!={\'A\',\'B\',\'C\'}:\n                    raise ValueError(\'Store all three road gates A, B and C before saving this three-arm camera.\')\n                record={**calibration[\'cameras\'][camera.value],\n                        \'gates\':[draft[n] for n in sorted(draft)],\'reviewed\':True}\n                core.validate_camera(record)\n                reference=folder/(camera.value+\'.png\')\n                if core.file_hash(reference)!=record[\'reference_sha256\']:\n                    raise ValueError(\'Reference image changed. Create a new calibration folder.\')\n                calibration[\'cameras\'][camera.value]=record\n                core.atomic_json(path,calibration)\n                print(\'Saved\',camera.value,\'to\',path)\n                print(\'A/B/C are road-arm identifiers. Place gates across each road arm, not across the grass.\')\n            except (ValueError,KeyError) as e:print(\'Not saved:\',e)\n\n    camera.observe(load_camera,names=\'value\');gate.observe(load_gate,names=\'value\')\n    for slider in sliders.values():slider.observe(render,names=\'value\')\n    label.observe(render,names=\'value\');store.on_click(store_gate);save.on_click(save_camera)\n    display(w.VBox([w.HTML(\'<b>For each road: move endpoints a/b across its traffic lanes. \'\n        \'Put the inside point toward the junction. Store A, B, C, then save the camera. \'\n        \'The red dots on pilot videos mark vehicle bottom-centres used for crossings.</b>\'),\n        w.HBox([camera,gate]),label,\n        w.HBox([w.VBox([sliders[n] for n in names[:3]]),w.VBox([sliders[n] for n in names[3:]])]),\n        w.HBox([store,save]),status,plot]))\n    load_camera()\n\n\ndef pilot_inventory(videos,cameras=core.CAMERAS,per_camera=3):\n    if per_camera<1:raise ValueError(\'Pilot clips per camera must be positive.\')\n    preferred=[\'normanniles2/normanniles2_2025-10-25-16-46-45.mp4\',\n               \'normanniles2/normanniles2_2025-10-25-17-33-45.mp4\',\n               \'normanniles2/normanniles2_2025-10-26-13-10-45.mp4\']\n    rank={key:i for i,key in enumerate(preferred)};selected=[]\n    for camera in cameras:\n        group=videos[videos.camera_id.eq(camera)].copy()\n        if group.empty:raise ValueError(\'No videos for \'+camera)\n        group[\'_rank\']=group.video_path.map(rank).fillna(100)\n        preferred_group=group[group._rank<100].sort_values(\'_rank\').head(per_camera)\n        rest=group[group._rank.eq(100)].sort_values(\'video_path\')\n        n=min(per_camera-len(preferred_group),len(rest))\n        spread=rest.iloc[np.linspace(0,len(rest)-1,n,dtype=int)] if n else rest.iloc[:0]\n        group=pd.concat([preferred_group,spread]).drop(columns=\'_rank\')\n        selected.append(group)\n    return pd.concat(selected).sort_values(\'video_path\').reset_index(drop=True)\n\n\ndef counting_signature(camera,detector,settings):\n    record={k:camera[k] for k in [\'gates\',\'reference_size\',\'reference_sha256\']}\n    record[\'clock_roi\']=camera.get(\'clock_roi\',[.68,0,1,.09])\n    return core.digest(dict(camera=record,detector=detector,settings={**core.DEFAULTS,**settings},\n        tracker=core.TRACKER,code_sha256=core.file_hash(core.__file__),\n        environment={n:importlib.metadata.version(n) for n in [\'ultralytics\',\'opencv-python\',\'numpy\',\'torch\']}))\n\n\ndef pilot_review_button(calibration_folder,pilot_run,pilot_videos,detector,settings):\n    import ipywidgets as w\n    from IPython.display import display,clear_output\n    calibration_folder=Path(calibration_folder);pilot_run=Path(pilot_run)\n    calibration=json.loads((calibration_folder/\'gates.json\').read_text())\n    contract=json.loads((pilot_run/\'contract.json\').read_text())\n    cameras=sorted(set(pilot_videos.camera_id))\n    if core.geometry(calibration,cameras)!=contract[\'config\'][\'geometry\']:\n        raise ValueError(\'Gates changed since these pilot videos were processed. Run a new pilot.\')\n    for r in pilot_videos.itertuples():\n        core.verify_checkpoint(core.clip_directory(pilot_run,r.video_path),contract[\'signature\'])\n        if not (core.clip_directory(pilot_run,r.video_path)/\'annotated.mp4\').is_file():\n            raise ValueError(\'Missing annotated pilot video.\')\n    out=w.Output()\n    button=w.Button(description=\'I reviewed pilot gates and directions\',button_style=\'success\',layout=w.Layout(width=\'350px\'))\n    def save(button):\n        path=calibration_folder/\'pilot_reviews.json\'\n        saved=json.loads(path.read_text()) if path.exists() else {}\n        # Reload so changing a gate after inference cannot approve stale previews.\n        now=json.loads((calibration_folder/\'gates.json\').read_text())\n        with out:\n            clear_output()\n            for camera in cameras:\n                if core.geometry(now,[camera])!=core.geometry(calibration,[camera]):\n                    print(\'Gates changed since these pilot previews. Run a new pilot.\');return\n                saved[detector+\'/\'+camera]=dict(counting_signature=counting_signature(now[\'cameras\'][camera],detector,settings),\n                    pilot_contract_signature=contract[\'signature\'],\n                    reviewed_utc=datetime.now(timezone.utc).isoformat(),\n                    video_paths=pilot_videos.loc[pilot_videos.camera_id.eq(camera),\'video_path\'].tolist())\n            core.atomic_json(path,saved)\n            print(\'Saved pilot geometry/direction review. Manual counting accuracy remains unmeasured until annotations are entered.\')\n    button.on_click(save)\n    display(w.HTML(\'Open the annotated pilot videos in Drive and inspect each gate, arrow and route. \'\n        \'This button records your visual geometry review; it does not certify counting accuracy.\'),button,out)\n\n\ndef require_pilot_reviews(folder,calibration,detector,settings):\n    path=Path(folder)/\'pilot_reviews.json\'\n    reviews=json.loads(path.read_text()) if path.exists() else {}\n    missing=[c for c in core.CAMERAS if reviews.get(detector+\'/\'+c,{}).get(\'counting_signature\')!=\n             counting_signature(calibration[\'cameras\'][c],detector,settings)]\n    if missing:raise ValueError(\'Run and visually review annotated pilot videos for: \'+\', \'.join(missing))\n\n\ndef export_bundle(run,calibration_folder,program_folder):\n    run=Path(run);calibration_folder=Path(calibration_folder);program_folder=Path(program_folder)\n    files={}\n    for name in [\'contract.json\',\'model_weights.json\',\'junction_features.csv\',\'pending_videos.csv\',\'processing_failures.csv\',\'batch_status.json\']:\n        p=run/name\n        if p.is_file():files[name]=p\n    if \'junction_features.csv\' not in files:raise ValueError(\'No completed feature export yet.\')\n    for name in [\'gates.json\',\'pilot_reviews.json\']:\n        p=calibration_folder/name\n        if p.is_file():files[\'calibration/\'+name]=p\n    for camera in core.CAMERAS:\n        p=calibration_folder/(camera+\'.png\')\n        if p.is_file():files[\'calibration/\'+p.name]=p\n    for name in [\'junction_counting_v4.py\',\'junction_calibration_v4.py\',\'junction_forecasting_v4.py\',\'junction_drawing_v4.py\']:\n        p=program_folder/name\n        if p.is_file():files[\'code/\'+name]=p\n    for folder in [\'evaluations\',\'manual_evaluation\']:\n        if (run/folder).exists():\n            for p in (run/folder).rglob(\'*\'):\n                if p.is_file():files[p.relative_to(run).as_posix()]=p\n    if (run/\'pilots\').exists():\n        for pilot in (run/\'pilots\').iterdir():\n            if not pilot.is_dir():continue\n            for name in [\'contract.json\',\'junction_features.csv\']:\n                p=pilot/name\n                if p.is_file():files[p.relative_to(run).as_posix()]=p\n            for folder in [\'manual_evaluation\',\'manual_annotations\']:\n                for p in (pilot/folder).rglob(\'*\'):\n                    if p.is_file():files[p.relative_to(run).as_posix()]=p\n    manifest={name:core.file_hash(p) for name,p in files.items()}\n    target=run/\'Junction_v4_Results.zip\';pending=target.with_suffix(\'.zip.pending\')\n    with zipfile.ZipFile(pending,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for name,p in files.items():z.write(p,name)\n        z.writestr(\'FILE_SHA256.json\',json.dumps(manifest,indent=2))\n        z.writestr(\'READ_ME.txt\',\'Predicted junction movements and optional forecasting outputs. \'\n            \'No counting accuracy is established without completed manual annotations. \'\n            \'Per-frame observations, quality flags, gate events, journeys and annotated pilot videos remain in the Drive run folders. \'\n            \'Complete journeys require both gates within a single clip. Counts are per camera; \'\n            \'overlapping views are not deduplicated across cameras. OCR/colour flags are candidates for review, not confirmed corruption.\\n\')\n    with zipfile.ZipFile(pending) as z:\n        if z.testzip() is not None:raise OSError(\'Archive integrity check failed.\')\n    pending.replace(target)\n    return target\n', 'junction_forecasting_v4.py': '"""Forecast congestion from calibrated journeys and past contiguous source clips."""\nfrom pathlib import Path\nimport json\nimport re\nimport hashlib\nimport importlib.metadata\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\nimport joblib\n\nfrom junction_counting_v4 import atomic_json, digest, file_hash, write_csv\n\nCLASSES=[\'free flowing\',\'heavy delay\',\'light delay\',\'moderate delay\']\nMEASUREMENTS=[\'complete_journeys\',\'incomplete_tracks\',\'observed_tracks\',\'gate_entries\',\'gate_exits\',\n    \'mean_vehicles\',\'max_vehicles\',\'density_std\',\'density_trend\',\'mean_bbox_area_fraction\',\n    \'bbox_occupancy_proxy\',\'mean_confidence\',\'motion_median\',\'motion_p90\',\'slow_fraction\',\n    \'weighted_complete_journeys\']\n\n\ndef validate_labels(labels):\n    required=[\'videos\',\'datetimestamp_start\',\'datetimestamp_end\',\'congestion_enter_rating\']\n    if set(required)-set(labels):raise ValueError(\'Label CSV is missing \'+str(set(required)-set(labels)))\n    t=labels[required].copy()\n    if t.isna().any().any():raise ValueError(\'Missing label/path/timestamp.\')\n    t[\'videos\']=t.videos.astype(str).str.replace(\'\\\\\',\'/\',regex=False).str.strip()\n    if not t.congestion_enter_rating.isin(CLASSES).all():raise ValueError(\'Unknown congestion label.\')\n    for c in [\'datetimestamp_start\',\'datetimestamp_end\']:\n        t[c]=pd.to_datetime(t[c],errors=\'raise\')\n    t=t.drop_duplicates()\n    if t.videos.duplicated().any():raise ValueError(\'Conflicting labels/timestamps for the same video.\')\n    if t.videos.eq(\'\').any():raise ValueError(\'Empty video path.\')\n    return t\n\n\ndef build_history(features,labels,history_clips=3):\n    """All candidates use the same eligible cohort. Missing history is never padded."""\n    if isinstance(history_clips,bool) or not isinstance(history_clips,int) or history_clips<1:\n        raise ValueError(\'history_clips must be a positive integer.\')\n    if features.video_path.duplicated().any():raise ValueError(\'Duplicate source features.\')\n    labels=validate_labels(labels)\n    table=features.copy()\n    # Only measured, explicit features are permitted; human labels never enter X.\n    numeric=MEASUREMENTS+sorted(c for c in table if re.fullmatch(\n        r\'(?:gate_[A-F]_(?:entry|exit)(?:_(?:car|motorcycle|bus|truck))?|route_[A-F]_[A-F](?:_(?:car|motorcycle|bus|truck))?|journeys_(?:car|motorcycle|bus|truck))\',c))\n    if set(MEASUREMENTS)-set(table):raise ValueError(\'Missing required junction measurements.\')\n    table[numeric]=table[numeric].apply(pd.to_numeric,errors=\'raise\').fillna(0)\n    table=table.merge(labels,on=None,left_on=\'video_path\',right_on=\'videos\',how=\'left\',validate=\'one_to_one\')\n    labelled=table.datetimestamp_start.notna()\n    parsed=table.video_path.str.extract(r\'_(\\d{4}-\\d{2}-\\d{2}-\\d{2}-\\d{2}-\\d{2})\\.mp4$\')[0]\n    fallback=pd.to_datetime(parsed,format=\'%Y-%m-%d-%H-%M-%S\',errors=\'coerce\')\n    table[\'source_time\']=table.datetimestamp_start.fillna(fallback)\n    table[\'segment_end\']=table.datetimestamp_end.fillna(table.source_time+pd.to_timedelta(table.duration_sec,unit=\'s\'))\n    table=table.dropna(subset=[\'source_time\',\'segment_end\']).sort_values([\'camera_id\',\'source_time\',\'video_path\']).reset_index(drop=True)\n    invalid=table.segment_end<=table.source_time\n    table=table[~invalid].copy().reset_index(drop=True)\n    if table.duplicated([\'camera_id\',\'source_time\']).any():raise ValueError(\'Ambiguous duplicate camera timestamps.\')\n    by_camera=labels.assign(camera_id=labels.videos.str.split(\'/\').str[0]).sort_values([\'camera_id\',\'datetimestamp_start\'])\n    by_camera=by_camera[by_camera.datetimestamp_end>by_camera.datetimestamp_start].copy()\n    g=by_camera.groupby(\'camera_id\')\n    by_camera[\'target_video_path\']=g.videos.shift(-1)\n    by_camera[\'target_time\']=g.datetimestamp_start.shift(-1)\n    by_camera[\'target_label\']=g.congestion_enter_rating.shift(-1)\n    by_camera[\'gap_seconds\']=(by_camera.target_time-by_camera.datetimestamp_end).dt.total_seconds()\n    pairs=by_camera[by_camera.gap_seconds.between(0,2)].set_index(\'videos\')\n    lookup=set(features.video_path)\n    x_current=[];x_history=[];metadata=[];excluded=[]\n    for camera,group in table.groupby(\'camera_id\',sort=False):\n        group=group.reset_index(drop=True)\n        for i,row in group.iterrows():\n            if row.video_path not in pairs.index:\n                excluded.append(dict(video_path=row.video_path,reason=\'no_valid_labelled_next_segment\'));continue\n            p=pairs.loc[row.video_path]\n            if p.target_video_path not in lookup:\n                excluded.append(dict(video_path=row.video_path,reason=\'target_video_not_processed\'));continue\n            if i<history_clips-1:\n                excluded.append(dict(video_path=row.video_path,reason=\'insufficient_prior_history\'));continue\n            window=group.iloc[i-history_clips+1:i+1]\n            gaps=(window.source_time.iloc[1:].reset_index(drop=True)-window.segment_end.iloc[:-1].reset_index(drop=True)).dt.total_seconds()\n            if not gaps.between(0,2).all():\n                excluded.append(dict(video_path=row.video_path,reason=\'history_is_not_contiguous\'));continue\n            if window.segment_end.max()>p.target_time:\n                raise ValueError(\'Source/history frames extend into the prediction target.\')\n            current={c:float(row[c]) for c in numeric}\n            hour=row.source_time.hour+row.source_time.minute/60\n            current.update(camera_id=camera,hour_sin=np.sin(2*np.pi*hour/24),hour_cos=np.cos(2*np.pi*hour/24))\n            hist=current.copy()\n            for lag in range(1,history_clips):\n                earlier=window.iloc[-lag-1]\n                for c in numeric:hist[f\'lag{lag}_{c}\']=float(earlier[c])\n            for c in numeric:\n                hist[\'past_mean_\'+c]=float(window[c].mean())\n                hist[\'past_change_\'+c]=float(window[c].iloc[-1]-window[c].iloc[0])\n            x_current.append(current);x_history.append(hist)\n            metadata.append(dict(video_path=row.video_path,camera_id=camera,source_time=row.source_time,\n                source_end=row.segment_end,history_start=window.source_time.iloc[0],\n                history_video_paths=json.dumps(window.video_path.tolist()),target_video_path=p.target_video_path,\n                target_time=p.target_time,target_label=p.target_label,\n                source_label=row.congestion_enter_rating))\n    return pd.DataFrame(metadata),pd.DataFrame(x_current),pd.DataFrame(x_history),pd.DataFrame(excluded)\n\n\ndef model_pipeline(kind,x):\n    cats=[\'camera_id\'];nums=[c for c in x if c not in cats]\n    prep=ColumnTransformer([\n        (\'numeric\',Pipeline([(\'fill\',SimpleImputer(strategy=\'median\',keep_empty_features=True)),(\'scale\',StandardScaler())]),nums),\n        (\'camera\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False),cats)])\n    if kind==\'logistic\':model=LogisticRegression(C=.3,max_iter=2000,random_state=42)\n    elif kind==\'forest\':model=RandomForestClassifier(n_estimators=250,max_depth=6,min_samples_leaf=3,max_features=.7,n_jobs=2,random_state=42)\n    elif kind==\'xgb\':model=XGBClassifier(n_estimators=150,max_depth=2,learning_rate=.04,min_child_weight=3,\n        reg_lambda=10,reg_alpha=.5,subsample=.85,colsample_bytree=.85,\n        objective=\'multi:softprob\',num_class=4,eval_metric=\'mlogloss\',tree_method=\'hist\',n_jobs=2,random_state=42)\n    else:raise ValueError(kind)\n    return Pipeline([(\'prep\',prep),(\'model\',model)])\n\n\ndef candidates():\n    return [(kind+\'_\'+family+suffix,kind,family,weight)\n        for family in [\'current\',\'history\']\n        for kind,suffix,weight in [(\'logistic\',\'_balanced\',\'balanced\'),(\'logistic\',\'_unweighted\',None),\n                                  (\'xgb\',\'_balanced\',\'balanced\'),(\'forest\',\'_balanced\',\'balanced\')]]\n\n\ndef scores(truth,pred):\n    return dict(n=len(truth),correct=int(np.sum(truth==pred)),accuracy=float(accuracy_score(truth,pred)),\n        balanced_accuracy=float(balanced_accuracy_score(truth,pred)),\n        macro_f1=float(f1_score(truth,pred,labels=range(4),average=\'macro\',zero_division=0)))\n\n\ndef evaluate(features,labels,output,test_start=\'2025-10-25\',test_end=\'2025-10-27\',history_clips=3):\n    output=Path(output)\n    meta,current,history,excluded=build_history(features,labels,history_clips)\n    if meta.empty:raise ValueError(\'No eligible contiguous histories; finish feature processing first.\')\n    start=pd.Timestamp(test_start);end=pd.Timestamp(test_end)\n    train=(meta.source_time<start)&(meta.target_time<start)\n    test=(meta.history_start>=start)&(meta.source_time>=start)&(meta.source_time<end)&(meta.target_time<end)\n    if not train.any() or not test.any():raise ValueError(\'No eligible chronological train/test split.\')\n    # Purge all target/test-frame overlap, not just duplicate source paths.\n    train_paths=set(meta.loc[train,\'video_path\'])|set(meta.loc[train,\'target_video_path\'])\n    test_paths=set(meta.loc[test,\'video_path\'])|set(meta.loc[test,\'target_video_path\'])\n    if train_paths&test_paths:raise ValueError(\'Train/test source or target overlap.\')\n    config=dict(code_sha256=file_hash(__file__),test_start=str(start),test_end=str(end),history_clips=history_clips,\n        features_sha256=hashlib.sha256(features.to_csv(index=False).encode()).hexdigest(),\n        labels_sha256=hashlib.sha256(labels.to_csv(index=False).encode()).hexdigest(),candidates=candidates(),\n        environment={n:importlib.metadata.version(n) for n in [\'scikit-learn\',\'xgboost\',\'numpy\',\'pandas\']})\n    signature=digest(config)\n    output.mkdir(parents=True,exist_ok=True)\n    contract=output/\'evaluation_contract.json\'\n    if contract.exists():\n        old=json.loads(contract.read_text())\n        if old[\'signature\']!=signature:raise ValueError(\'Evaluation inputs/settings changed; use a new EVALUATION_NAME.\')\n        if (output/\'evaluation_complete.json\').exists():\n            finished=json.loads((output/\'evaluation_complete.json\').read_text())\n            for name,expected in finished[\'sha256\'].items():\n                if not (output/name).is_file() or file_hash(output/name)!=expected:\n                    raise ValueError(\'Completed evaluation artifact changed: \'+name)\n            return pd.read_csv(output/\'evaluation_metrics.csv\')\n    else:atomic_json(contract,dict(signature=signature,config=config))\n    excluded.to_csv(output/\'excluded_pairs.csv\',index=False)\n    meta.assign(split=np.where(train,\'train\',np.where(test,\'test\',\'unused\'))).to_csv(output/\'selected_pairs.csv\',index=False)\n    x={\'current\':current,\'history\':history};y=meta.target_label.map({c:i for i,c in enumerate(CLASSES)}).to_numpy()\n    if set(y[train])!=set(range(4)):raise ValueError(\'Training must contain all four classes.\')\n    folds=[]\n    days=sorted(meta.loc[train,\'source_time\'].dt.normalize().unique())\n    for day in days[2:]:\n        day=pd.Timestamp(day)\n        fit=np.flatnonzero(train&(meta.target_time<day)&(meta.source_time<day))\n        valid=np.flatnonzero(train&(meta.history_start>=day)&(meta.source_time.dt.normalize()==day))\n        if len(fit) and len(valid) and set(y[fit])==set(range(4)):folds.append((day,fit,valid))\n    if len(folds)<2:raise ValueError(\'Need at least two usable chronological validation folds.\')\n    records=[]\n    for name,kind,family,weight in candidates():\n        for day,fit,valid in folds:\n            model=model_pipeline(kind,x[family]);weights=compute_sample_weight(weight,y[fit]) if weight else np.ones(len(fit))\n            model.fit(x[family].iloc[fit],y[fit],model__sample_weight=weights)\n            pred=model.predict(x[family].iloc[valid]);records.append(dict(candidate=name,validation_date=str(day),**scores(y[valid],pred)))\n    validation=pd.DataFrame(records);validation.to_csv(output/\'validation_folds.csv\',index=False)\n    ranking=validation.groupby(\'candidate\',as_index=False)[[\'macro_f1\',\'accuracy\']].mean().sort_values(\n        [\'macro_f1\',\'accuracy\',\'candidate\'],ascending=[False,False,True])\n    ranking.to_csv(output/\'validation_ranking.csv\',index=False)\n    name=ranking.iloc[0].candidate\n    selected=next(c for c in candidates() if c[0]==name)\n    family_winners={}\n    for family in [\'current\',\'history\']:\n        names={c[0] for c in candidates() if c[2]==family}\n        family_winners[family]=next(c for c in candidates() if c[0]==ranking[ranking.candidate.isin(names)].iloc[0].candidate)\n    selection=dict(selected_candidate=name,selection_metric=\'chronological training-date validation macro-F1\',\n        contract_signature=signature,evaluation_kind=\'previously_examined_week\' if start<pd.Timestamp(\'2025-10-27\') else \'new_dates_require_provenance_review\',\n        feature_names=x[selected[2]].columns.tolist(),family_winners={family:spec[0] for family,spec in family_winners.items()},\n        train_n=int(train.sum()),test_n=int(test.sum()))\n    atomic_json(output/\'selected_model.json\',selection)  # Before any test scoring.\n    fit=np.flatnonzero(train);test_idx=np.flatnonzero(test)\n    chosen=model_pipeline(selected[1],x[selected[2]])\n    weights=compute_sample_weight(selected[3],y[fit]) if selected[3] else np.ones(len(fit))\n    chosen.fit(x[selected[2]].iloc[fit],y[fit],model__sample_weight=weights)\n    predictions={\'selected_model\':chosen.predict(x[selected[2]].iloc[test_idx])}\n    # Frozen per-family winners give a same-test current/history ablation.\n    for family in [\'current\',\'history\']:\n        winner=family_winners[family]\n        model=model_pipeline(winner[1],x[family]);w=compute_sample_weight(winner[3],y[fit]) if winner[3] else np.ones(len(fit))\n        model.fit(x[family].iloc[fit],y[fit],model__sample_weight=w)\n        predictions[family+\'_validation_winner\']=model.predict(x[family].iloc[test_idx])\n    predictions[\'training_majority\']=np.full(len(test_idx),np.bincount(y[fit],minlength=4).argmax())\n    predictions[\'human_source_label_persistence\']=meta.loc[test,\'source_label\'].map({c:i for i,c in enumerate(CLASSES)}).to_numpy()\n    if pd.isna(predictions[\'human_source_label_persistence\']).any():raise ValueError(\'Missing source labels for reference baseline.\')\n    truth=y[test_idx];metrics=pd.DataFrame([dict(method=n,**scores(truth,p)) for n,p in predictions.items()])\n    metrics.to_csv(output/\'evaluation_metrics.csv\',index=False)\n    export=meta.loc[test].copy()\n    for n,p in predictions.items():export[n]=[CLASSES[int(k)] for k in p]\n    export.to_csv(output/\'evaluation_predictions.csv\',index=False)\n    matrix=pd.DataFrame(confusion_matrix(truth,predictions[\'selected_model\'],labels=range(4)),index=CLASSES,columns=CLASSES)\n    matrix.to_csv(output/\'confusion_matrix.csv\')\n    pd.DataFrame(classification_report(truth,predictions[\'selected_model\'],labels=range(4),target_names=CLASSES,\n        output_dict=True,zero_division=0)).T.to_csv(output/\'classification_report.csv\')\n    distribution=pd.DataFrame({\'class\':CLASSES,\'train\':[int(sum(y[fit]==i)) for i in range(4)],\n                               \'test\':[int(sum(truth==i)) for i in range(4)]})\n    distribution.to_csv(output/\'split_class_distribution.csv\',index=False)\n    joblib.dump(chosen,output/\'selected_model.joblib\')\n    (output/\'REPORT.md\').write_text(\'# Junction forecasting experiment\\n\\n\'\n        f\'Selected using training-date validation: `{name}`.\\n\\n\'\n        f\'Evaluation provenance: `{selection["evaluation_kind"]}`. The default October 2025 week was examined before.\\n\\n\'\n        \'Current and history candidates use the same eligible source/target pairs and test sample. \'\n        \'Quality flags are retained for review; no test clips are removed based on their errors. \'\n        \'Persistence copies a human-provided current label and is a reference baseline, not an input available to the predictor. \'\n        \'Scores assess next-segment congestion, not vehicle-counting accuracy.\\n\\n\'+metrics.to_string(index=False)+\'\\n\',encoding=\'utf-8\')\n    atomic_json(output/\'evaluation_complete.json\',dict(contract_signature=signature,\n        sha256={p.name:file_hash(p) for p in output.iterdir() if p.is_file() and p.name not in\n                {\'evaluation_complete.json\',\'confusion_matrix.png\'}}))\n    return metrics\n\n\ndef manual_templates(features,calibration,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    routes=[];gates=[]\n    for r in features.itertuples():\n        names=sorted(g[\'name\'] for g in calibration[\'cameras\'][r.camera_id][\'gates\'])\n        for a in names:\n            for cls in [\'car\',\'motorcycle\',\'bus\',\'truck\']:\n                for direction in [\'entry\',\'exit\']:\n                    gates.append(dict(video_path=r.video_path,gate=a,event=direction,vehicle_class=cls,\n                                      ground_truth_count=\'\',annotation_status=\'pending\'))\n                for b in names:\n                    if a!=b:routes.append(dict(video_path=r.video_path,origin=a,destination=b,vehicle_class=cls,\n                                               ground_truth_count=\'\',annotation_status=\'pending\'))\n    for name,rows in [(\'manual_journey_counts.csv\',routes),(\'manual_gate_counts.csv\',gates)]:\n        path=output/name\n        if not path.exists() and rows:write_csv(path,rows,list(rows[0]))\n\n\ndef evaluate_manual_counts(features,calibration,annotation_file,output,kind=\'journeys\'):\n    """Reject partial annotations. Compare only fully annotated full clips."""\n    t=pd.read_csv(annotation_file,keep_default_na=False)\n    keys=[\'video_path\',\'origin\',\'destination\',\'vehicle_class\'] if kind==\'journeys\' else [\'video_path\',\'gate\',\'event\',\'vehicle_class\']\n    if set(keys+[\'ground_truth_count\',\'annotation_status\'])-set(t):raise ValueError(\'Missing annotation columns.\')\n    if t.empty or t.duplicated(keys).any():raise ValueError(\'Empty or duplicate annotations.\')\n    if not t.annotation_status.eq(\'complete\').all():raise ValueError(\'All submitted rows must be marked complete; pending/uncertain counts cannot be scored.\')\n    counts=pd.to_numeric(t.ground_truth_count,errors=\'raise\')\n    if counts.isna().any() or ((counts<0)|(counts%1!=0)).any():raise ValueError(\'Enter a nonnegative integer for every count, including zero.\')\n    lookup=features.set_index(\'video_path\');rows=[]\n    for video,group in t.groupby(\'video_path\'):\n        if video not in lookup.index:raise ValueError(\'No processed feature row for \'+video)\n        row=lookup.loc[video];names=sorted(g[\'name\'] for g in calibration[\'cameras\'][row.camera_id][\'gates\'])\n        classes=[\'car\',\'motorcycle\',\'bus\',\'truck\']\n        expected={(a,b,c) for a in names for b in names if a!=b for c in classes} if kind==\'journeys\' else {\n            (a,d,c) for a in names for d in [\'entry\',\'exit\'] for c in classes}\n        actual=set(map(tuple,group[keys[1:]].to_numpy()))\n        if actual!=expected:raise ValueError(\'Annotate every permitted route/gate and class, including zeros, for \'+video)\n        for _,r in group.iterrows():\n            if kind==\'journeys\':\n                col=f\'route_{r.origin}_{r.destination}_{r.vehicle_class}\'\n                details=dict(origin=r.origin,destination=r.destination)\n            elif kind==\'gates\':\n                col=f\'gate_{r.gate}_{r.event}_{r.vehicle_class}\';details=dict(gate=r.gate,event=r.event)\n            else:raise ValueError(\'kind must be journeys or gates.\')\n            rows.append(dict(video_path=video,**details,vehicle_class=r.vehicle_class,\n                             predicted=int(row[col]),ground_truth=int(r.ground_truth_count)))\n    result=pd.DataFrame(rows);error=result.predicted-result.ground_truth\n    totals=result.groupby(\'video_path\')[[\'predicted\',\'ground_truth\']].sum();te=totals.predicted-totals.ground_truth\n    report=dict(annotation_kind=kind,annotated_clips=len(totals),cell_mae=float(abs(error).mean()),\n                cell_rmse=float(np.sqrt((error**2).mean())),\n                clip_total_mae=float(abs(te).mean()),clip_total_rmse=float(np.sqrt((te**2).mean())))\n    output=Path(output);output.mkdir(parents=True,exist_ok=True);result.to_csv(output/\'manual_comparison.csv\',index=False)\n    atomic_json(output/\'manual_metrics.json\',report)\n    return report\n', 'junction_drawing_v4.py': '"""Self-contained browser drawing: two endpoints and one junction-side click."""\nfrom copy import deepcopy\nfrom pathlib import Path\nimport base64\nimport json\n\nimport junction_counting_v4 as core\n\n\nDRAWING_HTML = r\'\'\'<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Draw camera gates</title><style>\n*{box-sizing:border-box}body{margin:0;background:#edf2f7;color:#172338;font:16px system-ui,Segoe UI,sans-serif}\nmain{max-width:1250px;margin:auto;padding:22px}h1{font-size:28px;margin:0 0 8px}p{line-height:1.5}\n.bar{display:flex;gap:12px;align-items:center;flex-wrap:wrap;background:white;border-radius:12px;padding:14px;margin:12px 0}\nlabel{font-weight:600}select,input,button{font:inherit;padding:9px 12px;border:1px solid #c6d1df;border-radius:7px}\nbutton{background:white;cursor:pointer}button.primary{background:#17633f;color:white;border-color:#17633f}button:disabled{opacity:.45;cursor:not-allowed}\nselect{min-width:140px}input{max-width:220px}.step{background:#102a43;color:white;padding:14px;border-radius:10px;font-size:19px;font-weight:600}\ncanvas{display:block;width:100%;height:auto;max-height:none;background:#111;outline:1px solid #a9b8ca;touch-action:none;cursor:crosshair}\n.picture{margin:14px 0;box-shadow:0 4px 18px #17233815}.status{min-height:28px;margin:12px 0;color:#154d32;font-weight:600}.error{color:#a51d29}\n.tag{background:#e6eff8;padding:5px 9px;border-radius:7px}small{color:#536278}.hint{margin:6px 0 16px}.legend{font-size:15px}\n</style></head><body><main>\n<h1>Draw your road gates with the mouse</h1>\n<p class="hint">For each road, click <b>one end</b>, click <b>the other end</b>, then click <b>toward the junction</b>.\nYou can also drag a line, release it, then click its junction side. Repeat for A, B and C.</p>\n<div class="bar"><label>Camera <select id="camera"></select></label><label>Gate <select id="gate"><option>A</option><option>B</option><option>C</option></select></label>\n<label>Road name <input id="road" placeholder="e.g. left road"></label><button id="redraw">Redraw this gate</button><button id="undo">Undo last point</button></div>\n<div id="step" class="step" aria-live="polite"></div><div class="picture"><canvas id="canvas" width="960" height="540" aria-label="Click the image to draw the selected gate"></canvas></div>\n<div class="bar"><span class="legend">Yellow = counting gate &nbsp; Green dot/arrow = junction side</span>\n<button id="save" class="primary">Save this camera</button><span id="progress" class="tag"></span></div>\n<div class="bar"><button id="download" class="primary">Download gates_drawn.json</button>\n<label>Resume a saved drawing <input id="resume" type="file" accept=".json,application/json"></label>\n<button id="draft">Download unfinished draft</button></div>\n<div id="status" class="status" role="status"></div>\n<p>After saving every camera, download <b>gates_drawn.json</b>. Return to the notebook and use its <b>Import drawn gates</b> control.\nDo not count vehicles by drawing a triangle over the grass. Put one gate across each road arm and leave visible road on both sides.</p>\n<small id="scope"></small>\n</main><script type="application/json" id="payload">__PAYLOAD__</script><script>\n\'use strict\';\nconst payload=JSON.parse(document.getElementById(\'payload\').textContent);\nconst calibration=structuredClone(payload.calibration), ids=Object.keys(calibration.cameras);\nconst camera=document.getElementById(\'camera\'), gate=document.getElementById(\'gate\'), road=document.getElementById(\'road\');\nconst canvas=document.getElementById(\'canvas\'),ctx=canvas.getContext(\'2d\'),status=document.getElementById(\'status\');\nlet image=null,points=[],pressed=null,hover=null,loadVersion=0,dirty=false;\nids.forEach(id=>{const option=document.createElement(\'option\');option.value=id;option.textContent=id;camera.append(option)});\nfunction record(){return calibration.cameras[camera.value]}\nfunction selected(){return record().gates.find(g=>g.name===gate.value)}\nfunction notice(text,error=false){status.textContent=text;status.className=\'status\'+(error?\' error\':\'\')}\nfunction product(a,b,p){return (b[0]-a[0])*(p[1]-a[1])-(b[1]-a[1])*(p[0]-a[0])}\nfunction crossing(a,b,p,q){const r=[q[0]-p[0],q[1]-p[1]],s=[b[0]-a[0],b[1]-a[1]],den=r[0]*s[1]-r[1]*s[0];\n if(Math.abs(den)<1e-12)return false;const d=[a[0]-p[0],a[1]-p[1]],t=(d[0]*s[1]-d[1]*s[0])/den,u=(d[0]*r[1]-d[1]*r[0])/den;\n return t>=0&&t<=1&&u>=0&&u<=1}\nfunction validGate(g){for(const field of [\'a\',\'b\',\'inside\']){if(!Array.isArray(g[field])||g[field].length!==2||g[field].some(x=>!Number.isFinite(x)||x<0||x>1))return \'Click all three points on the picture.\'}\n const length=Math.hypot(g.a[0]-g.b[0],g.a[1]-g.b[1]);\n if(length<.02)return \'The gate is too short. Choose endpoints across the road.\';\n if(Math.abs(product(g.a,g.b,g.inside))/length<.005)return \'Click farther from the yellow line, on the junction side.\';return \'\'}\nfunction cameraError(r){const names=r.gates.map(g=>g.name).sort().join(\'\');if(names!==\'ABC\')return \'Draw gates A, B and C before saving this camera.\';\n for(const g of r.gates){const problem=validGate(g);if(problem)return g.name+\': \'+problem}\n for(let i=0;i<r.gates.length;i++)for(let k=i+1;k<r.gates.length;k++)if(crossing(r.gates[i].a,r.gates[i].b,r.gates[k].a,r.gates[k].b))return \'Gates intersect. Place them separately across the road arms.\';\n return \'\'}\nfunction xy(p){return [p[0]*canvas.width,p[1]*canvas.height]}\nfunction text(t,x,y,colour){ctx.font=Math.max(14,canvas.width/55)+\'px system-ui\';ctx.lineWidth=4;ctx.strokeStyle=\'#102030\';ctx.strokeText(t,x,y);ctx.fillStyle=colour;ctx.fillText(t,x,y)}\nfunction dot(p,label,colour){const [x,y]=xy(p);ctx.beginPath();ctx.arc(x,y,Math.max(4,canvas.width/180),0,Math.PI*2);ctx.fillStyle=colour;ctx.fill();text(label,x+8,y-8,colour)}\nfunction line(a,b,colour){const [x,y]=xy(a),[u,v]=xy(b);ctx.beginPath();ctx.moveTo(x,y);ctx.lineTo(u,v);ctx.lineWidth=Math.max(3,canvas.width/260);ctx.strokeStyle=colour;ctx.stroke()}\nfunction drawGate(g){line(g.a,g.b,\'#ffe600\');dot(g.a,g.name+\' a\',\'#ffe600\');dot(g.b,\'b\',\'#ffe600\');\n const middle=[(g.a[0]+g.b[0])/2,(g.a[1]+g.b[1])/2];line(middle,g.inside,\'#36ef66\');\n const [x,y]=xy(middle),[u,v]=xy(g.inside),angle=Math.atan2(v-y,u-x),len=Math.max(10,canvas.width/70);\n ctx.beginPath();ctx.moveTo(u,v);ctx.lineTo(u-len*Math.cos(angle-.5),v-len*Math.sin(angle-.5));ctx.lineTo(u-len*Math.cos(angle+.5),v-len*Math.sin(angle+.5));ctx.closePath();ctx.fillStyle=\'#36ef66\';ctx.fill();dot(g.inside,\'inside\',\'#36ef66\')}\nfunction render(){ctx.clearRect(0,0,canvas.width,canvas.height);if(image)ctx.drawImage(image,0,0,canvas.width,canvas.height);\n for(const g of record().gates)if(!(g.name===gate.value&&points.length))drawGate(g);\n if(points.length){dot(points[0],gate.value+\' a\',\'#ffe600\');if(points[1]){line(points[0],points[1],\'#ffe600\');dot(points[1],\'b\',\'#ffe600\')}else if(hover)line(points[0],hover,\'#ffe600\')}\n if(pressed&&pressed.phase===0&&hover)line(pressed.point,hover,\'#ffe600\');\n const ready=Boolean(selected())&&points.length===0;\n document.getElementById(\'step\').textContent=!image?\'Loading reference picture…\':ready?\'Gate \'+gate.value+\' is drawn. Choose the next gate, or Save this camera.\':\n [\'1 — Click the first end of gate \'+gate.value+\' (or drag across the road).\',\'2 — Click the other end of gate \'+gate.value+\'.\',\'3 — Click on the junction side of the yellow line.\'][points.length];\n const saved=ids.filter(id=>calibration.cameras[id].reviewed&&!cameraError(calibration.cameras[id])).length;\n document.getElementById(\'progress\').textContent=saved+\' / \'+ids.length+\' cameras saved\';document.getElementById(\'download\').disabled=saved!==ids.length}\nfunction loadGate(){points=[];pressed=null;hover=null;road.value=selected()?.label||\'\';render()}\nfunction loadCamera(){const version=++loadVersion;image=null;const r=record();canvas.width=r.reference_size[0];canvas.height=r.reference_size[1];\n loadGate();const next=new Image();next.onload=()=>{if(version!==loadVersion)return;image=next;render()};next.onerror=()=>notice(\'Could not load this reference picture.\',true);next.src=payload.images[camera.value]}\nfunction normalized(event){const rect=canvas.getBoundingClientRect();return [Math.max(0,Math.min(1,(event.clientX-rect.left)/rect.width)),Math.max(0,Math.min(1,(event.clientY-rect.top)/rect.height))]}\nfunction changed(){record().reviewed=false;dirty=true}\nfunction accept(point){if(!image)return;if(selected()&&points.length===0){notice(\'This gate is already drawn. Choose Redraw this gate to change it.\');return}\n if(points.length<2){points.push(point);changed();notice(\'\');render();return}\n const g={name:gate.value,label:road.value.trim(),a:points[0],b:points[1],inside:point};const problem=validGate(g);\n if(problem){notice(problem,true);return}record().gates=record().gates.filter(old=>old.name!==g.name);record().gates.push(g);changed();points=[];hover=null;notice(\'Gate \'+gate.value+\' drawn. Select the next gate, or save this camera.\');render()}\ncanvas.addEventListener(\'pointerdown\',event=>{if(!image)return;pressed={point:normalized(event),phase:points.length,canDraw:!selected()||points.length>0};canvas.setPointerCapture(event.pointerId)});\ncanvas.addEventListener(\'pointermove\',event=>{hover=normalized(event);render()});\ncanvas.addEventListener(\'pointerup\',event=>{if(!pressed)return;const start=pressed,end=normalized(event);pressed=null;\n const distance=Math.hypot((end[0]-start.point[0])*canvas.width,(end[1]-start.point[1])*canvas.height);\n if(start.phase===0&&start.canDraw&&distance>=Math.max(6,canvas.width*.02)){points=[start.point,end];changed();notice(\'Line drawn. Now click its junction side.\');render()}else accept(end)});\ncanvas.addEventListener(\'pointercancel\',()=>{pressed=null;hover=null;render()});\ndocument.getElementById(\'redraw\').onclick=()=>{record().gates=record().gates.filter(g=>g.name!==gate.value);points=[];hover=null;changed();notice(\'Draw gate \'+gate.value+\' again.\');render()};\ndocument.getElementById(\'undo\').onclick=()=>{if(points.length){points.pop();changed()}else if(selected()){const g=selected();points=[g.a,g.b];record().gates=record().gates.filter(x=>x.name!==gate.value);changed()}notice(\'\');render()};\nroad.oninput=()=>{if(selected()){selected().label=road.value.trim();changed();render()}};\nfunction unfinished(){return points.length>0}\nfunction switchTo(element,oldValue){if(unfinished()&&!confirm(\'Discard the unfinished points for this gate?\')){element.value=oldValue;return false}return true}\nlet previousCamera=camera.value,previousGate=gate.value;\ncamera.onchange=()=>{if(!switchTo(camera,previousCamera))return;previousCamera=camera.value;notice(\'\');loadCamera()};\ngate.onchange=()=>{if(!switchTo(gate,previousGate))return;previousGate=gate.value;notice(\'\');loadGate()};\ndocument.getElementById(\'save\').onclick=()=>{if(unfinished()){notice(\'Finish the current gate by clicking its junction side first.\',true);return}\n const error=cameraError(record());if(error){notice(error,true);return}record().reviewed=true;dirty=true;notice(camera.value+\' saved in this drawing. Save all cameras, then download the JSON file.\');render()};\nfunction download(complete){if(complete&&ids.some(id=>!calibration.cameras[id].reviewed||cameraError(calibration.cameras[id]))){notice(\'Save every camera before downloading the completed gates.\',true);return}\n const blob=new Blob([JSON.stringify(calibration,null,2)],{type:\'application/json\'}),url=URL.createObjectURL(blob),a=document.createElement(\'a\');\n a.href=url;a.download=complete?\'gates_drawn.json\':\'gates_draft.json\';a.click();dirty=false;setTimeout(()=>URL.revokeObjectURL(url),1000);notice(complete?\'Downloaded gates_drawn.json. Import it in the notebook.\':\'Downloaded an unfinished draft. Use Resume a saved drawing to continue it later.\')}\ndocument.getElementById(\'download\').onclick=()=>download(true);document.getElementById(\'draft\').onclick=()=>download(false);\ndocument.getElementById(\'resume\').onchange=async event=>{const file=event.target.files[0];if(!file)return;try{const draft=JSON.parse(await file.text());\n if(Object.keys(draft.cameras||{}).sort().join(\'|\')!==ids.slice().sort().join(\'|\'))throw Error(\'Camera list does not match this drawing page.\');\n for(const id of ids){const r=draft.cameras[id],original=payload.calibration.cameras[id];if(r.reference_sha256!==original.reference_sha256||JSON.stringify(r.reference_size)!==JSON.stringify(original.reference_size))throw Error(\'Reference picture differs for \'+id);\n const names=new Set();for(const g of r.gates||[]){if(![\'A\',\'B\',\'C\'].includes(g.name)||names.has(g.name)||validGate(g))throw Error(\'Invalid draft gate for \'+id);names.add(g.name)}\n if(r.reviewed&&cameraError(r))throw Error(\'Saved camera contains incomplete/intersecting gates: \'+id)}\n for(const id of ids){calibration.cameras[id].gates=draft.cameras[id].gates;calibration.cameras[id].reviewed=Boolean(draft.cameras[id].reviewed)}\n dirty=false;notice(\'Drawing restored.\');loadCamera()}catch(error){notice(error.message,true)}finally{event.target.value=\'\'}};\ndocument.getElementById(\'scope\').textContent=\'Images and drawing stay in this browser. \'+(payload.note||\'Use the original reference pictures and visually review annotated pilots before batch processing.\');\nwindow.addEventListener(\'beforeunload\',event=>{if(dirty||unfinished()){event.preventDefault();event.returnValue=\'\'}});\nwindow.gateDrawing={getCalibration:()=>structuredClone(calibration),cameraError,validGate};\nloadCamera();\n</script></body></html>\'\'\'\n\n\ndef write_drawing_page(folder):\n    folder=Path(folder)\n    calibration=json.loads((folder/\'gates.json\').read_text(encoding=\'utf-8\'))\n    images={}\n    for name,camera in calibration[\'cameras\'].items():\n        image=folder/(name+\'.png\')\n        if core.file_hash(image)!=camera[\'reference_sha256\']:\n            raise ValueError(\'Reference picture changed: \'+name)\n        images[name]=\'data:image/png;base64,\'+base64.b64encode(image.read_bytes()).decode(\'ascii\')\n    payload=json.dumps(dict(calibration=calibration,images=images),ensure_ascii=True).replace(\'<\',chr(92)+\'u003c\')\n    target=folder/\'Draw_Camera_Gates.html\'\n    target.write_text(DRAWING_HTML.replace(\'__PAYLOAD__\',payload),encoding=\'utf-8\')\n    return target\n\n\ndef import_drawn_gates(folder,uploaded):\n    """Import geometry only, bound to the original reference hashes and dimensions."""\n    folder=Path(folder);path=folder/\'gates.json\'\n    original=json.loads(path.read_text(encoding=\'utf-8\'))\n    if isinstance(uploaded,(bytes,bytearray,memoryview)):\n        drawn=json.loads(bytes(uploaded).decode(\'utf-8-sig\'))\n    elif isinstance(uploaded,str):drawn=json.loads(uploaded)\n    else:drawn=deepcopy(uploaded)\n    if not isinstance(drawn,dict) or set(drawn.get(\'cameras\',{}))!=set(original[\'cameras\']):\n        raise ValueError(\'Drawn gates must contain the same camera references.\')\n    result=deepcopy(original)\n    for name,old in original[\'cameras\'].items():\n        record=drawn[\'cameras\'][name]\n        if any(record.get(k)!=old[k] for k in [\'reference_size\',\'reference_sha256\']):\n            raise ValueError(\'Drawing uses another reference image: \'+name)\n        if core.file_hash(folder/(name+\'.png\'))!=old[\'reference_sha256\']:\n            raise ValueError(\'Original reference picture changed: \'+name)\n        if {g.get(\'name\') for g in record.get(\'gates\',[])}!={\'A\',\'B\',\'C\'}:\n            raise ValueError(\'Draw all three gates A, B and C for \'+name)\n        if record.get(\'reviewed\') is not True:raise ValueError(\'Save this camera in the drawing page first: \'+name)\n        result[\'cameras\'][name].update(gates=deepcopy(record[\'gates\']),reviewed=True)\n        core.validate_camera(result[\'cameras\'][name])\n    # Preserve the previous calibration once per distinct geometry, including\n    # pilot-compatible positions, rather than overwriting it without a copy.\n    backup=folder/(\'gates_before_drawing_\'+core.digest(original)[:12]+\'.json\')\n    if result!=original and not backup.exists():core.atomic_json(backup,original)\n    core.atomic_json(path,result)\n    return result\n\n\ndef drawing_import_widget(folder):\n    import ipywidgets as w\n    from IPython.display import display,clear_output\n    upload=w.FileUpload(accept=\'.json\',multiple=False,description=\'Choose gates JSON\')\n    button=w.Button(description=\'Import drawn gates\',button_style=\'success\')\n    out=w.Output()\n    def load(button):\n        with out:\n            clear_output()\n            if not upload.value:print(\'Choose the downloaded gates_drawn.json file first.\');return\n            try:\n                value=upload.value\n                item=next(iter(value.values())) if isinstance(value,dict) else value[0]\n                result=import_drawn_gates(folder,item[\'content\'])\n                print(\'Imported reviewed gates for\',len(result[\'cameras\']),\'cameras. Now run the pilot stage.\')\n            except (ValueError,KeyError,TypeError,OSError) as error:print(\'Not imported:\',error)\n    button.on_click(load)\n    display(w.HTML(\'<b>After drawing:</b> choose gates_drawn.json below, then click Import drawn gates.\'),upload,button,out)\n'}
for filename,source in MODULE_SOURCES.items():
    (PROGRAM_FOLDER/filename).write_text(source,encoding='utf-8')
sys.path.insert(0,str(PROGRAM_FOLDER))
for module_name in ['junction_counting_v4','junction_calibration_v4','junction_forecasting_v4','junction_drawing_v4']:
    sys.modules.pop(module_name,None)
import junction_counting_v4 as junction
import junction_calibration_v4 as calibration_tools
import junction_forecasting_v4 as forecasting
import junction_drawing_v4 as drawing_tools
import pandas as pd
import json
from IPython.display import display

Mounted at /content/drive


In [11]:
CALIBRATION_METHOD='draw'  # Mouse drawing; use 'sliders' for the previous editor.
# Cell 3 — configuration. Keep names/settings unchanged when resuming.
DRIVE_BASE=Path('/content/drive/MyDrive/CP-1 Dataset')
STAGE='pilot'  # calibrate, pilot, batch, evaluate, export
DETECTOR='yolo11'  # yolo11 or yolov8; each has independent outputs and pilot reviews
EXPERIMENT_NAME='junction_01'
CALIBRATION_NAME='camera_gates_01'
EVALUATION_NAME='historical_week_01'
PILOT_CAMERAS=list(junction.CAMERAS)  # Can use ['normanniles2'] for an initial camera-2-only pilot.
PILOT_CLIPS_PER_CAMERA=3
HISTORY_CLIPS=3
TEST_START='2025-10-25'
TEST_END_EXCLUSIVE='2025-10-27'
LABELS_PATH=DRIVE_BASE/'CSV'/'Train.csv'
SETTINGS=dict(junction.DEFAULTS)
# OCR is sampled once per second; its flags require review and never exclude a clip automatically.
# Set clock_sample_seconds=0 to disable both sampled image/clock checks before starting a NEW experiment.
assert STAGE in {'calibrate','pilot','batch','evaluate','export'}
if STAGE in {'pilot','batch'}:
    import torch
    if not torch.cuda.is_available():raise RuntimeError('Select a Colab GPU runtime before pilot/batch inference.')
    print('GPU:',torch.cuda.get_device_name(0))
for name in [EXPERIMENT_NAME,CALIBRATION_NAME,EVALUATION_NAME]:
    if not junction.re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*',name):raise ValueError('Use simple folder names.')
CALIBRATION_FOLDER=DRIVE_BASE/'Junction_v4'/'calibration'/CALIBRATION_NAME
RUN_FOLDER=DRIVE_BASE/'Junction_v4'/DETECTOR/EXPERIMENT_NAME
RUN_FOLDER.mkdir(parents=True,exist_ok=True)

GPU: Tesla T4


In [12]:
# Cell 4 — inventory all videos in the four camera folders, then get actual reference frames.
VIDEOS=junction.inventory(DRIVE_BASE)
display(VIDEOS.groupby('camera_id').agg(videos=('video_path','size'),bytes=('size_bytes','sum')))
print('Total planned videos:',len(VIDEOS))
print('Calibration folder:',CALIBRATION_FOLDER)
print('Result folder:',RUN_FOLDER)
CALIBRATION=junction.reference_frames(DRIVE_BASE,VIDEOS,CALIBRATION_FOLDER)
if STAGE=='calibrate':
    if CALIBRATION_METHOD=='draw':
        drawing_page=drawing_tools.write_drawing_page(CALIBRATION_FOLDER)
        print('Download this HTML from Google Drive and open it in Chrome/Edge:',drawing_page)
        print('Draw gates with the mouse; download gates_drawn.json, then import it below.')
        drawing_tools.drawing_import_widget(CALIBRATION_FOLDER)
    elif CALIBRATION_METHOD=='sliders':
        calibration_tools.gate_editor(CALIBRATION_FOLDER)
    else:raise ValueError('CALIBRATION_METHOD must be draw or sliders.')
else:
    CALIBRATION=json.loads((CALIBRATION_FOLDER/'gates.json').read_text())
    needed=PILOT_CAMERAS if STAGE=='pilot' else junction.CAMERAS
    junction.geometry(CALIBRATION,needed)

,videos,bytes
camera_id,,
normanniles1,4633,35705846066
normanniles2,4633,37428848447
normanniles3,4633,35647671068
normanniles4,4633,34494313518


Total planned videos: 18532
Calibration folder: /content/drive/MyDrive/CP-1 Dataset/Junction_v4/calibration/camera_gates_01
Result folder: /content/drive/MyDrive/CP-1 Dataset/Junction_v4/yolo11/junction_01


### Gate placement and manual counting

Name the visible road arms A/B/C (e.g. left, upper, right for camera 2). Move each gate's endpoints **across that road arm**, spanning its entry and exit traffic, without intersecting the other gates. Put the green-arrow `inside` point on the junction side. The algorithm uses the **bottom centre of each predicted vehicle box**; the pilot marks it with a red dot. Confirm that real inbound vehicles produce `entry` and outbound vehicles produce `exit` at the correct gate.

With the mouse drawing page: choose camera and gate; click endpoint 1, click endpoint 2, click toward the junction. Alternatively drag the yellow line, release, then click toward the junction. Use Undo or Redraw to fix a gate. Click Save this camera after all A/B/C gates are drawn. The page embeds the real reference images and works offline; return the downloaded JSON through the notebook's import control.

The existing camera-1 glitches and camera-2 colour shift remain review observations, not reasons to selectively discard misclassified clips. Review flags and labelled/correctly classified clips as well. A colour change alone does not make a manual count invalid.

In [ ]:
# Cell 6 — short annotated pilot and manual-annotation templates; no full batch here.
if STAGE=='pilot':
    PILOT_VIDEOS=calibration_tools.pilot_inventory(VIDEOS,PILOT_CAMERAS,PILOT_CLIPS_PER_CAMERA)
    pilot_id=junction.digest(dict(videos=PILOT_VIDEOS.video_path.tolist(),
        geometry=junction.geometry(CALIBRATION,PILOT_CAMERAS),settings=SETTINGS,
        code=junction.file_hash(junction.__file__)))[:12]
    PILOT_FOLDER=RUN_FOLDER/'pilots'/pilot_id
    PILOT_FEATURES,PILOT_PENDING=junction.run_batch(DRIVE_BASE,PILOT_FOLDER,PILOT_VIDEOS,
        CALIBRATION,DETECTOR,SETTINGS,preview=True)
    if PILOT_PENDING:raise RuntimeError('Some pilot clips failed. Resolve processing_failures.csv and resume.')
    forecasting.manual_templates(PILOT_FEATURES,CALIBRATION,PILOT_FOLDER/'manual_annotations')
    display(PILOT_FEATURES[['video_path','gate_entries','gate_exits','complete_journeys','incomplete_tracks','quality_flags']])
    print('Open each annotated.mp4 under:',PILOT_FOLDER/'clips')
    manifests=[json.loads((junction.clip_directory(PILOT_FOLDER,r.video_path)/'manifest.json').read_text())
               for r in PILOT_VIDEOS.itertuples()]
    seconds=[m['processing_seconds'] for m in manifests]
    estimate=sum(seconds)/len(seconds)*len(VIDEOS)/3600
    print(f'Rough full-run estimate from this pilot: {estimate:.1f} GPU/runtime hours; I/O and scene load can change it.')
    calibration_tools.pilot_review_button(CALIBRATION_FOLDER,PILOT_FOLDER,PILOT_VIDEOS,DETECTOR,SETTINGS)
else:
    print('Pilot runs only with STAGE="pilot". Save calibrated gates first.')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[1/12] Processing: normanniles1/normanniles1_2025-10-20-06-00-45.mp4
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 260ms
Prepared 1 package in 37ms
Installed 1 package in 3ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  Complete journeys: 0; incomplete tracks: 9; new-clip mean: 2.72 min
[2/12] Processing: normanniles1/normanniles1_2025-10-23-11-49-45.mp4
  Complete journeys: 0; incomplete tracks: 101; new-clip mean: 2.64 min
[3/12] Processing: normanniles1/normanniles1_2025-10-26-17-59-45.mp4
 

,video_path,gate_entries,gate_exits,complete_journeys,incomplete_tracks,quality_flags
0,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,5,0,0,9,0
1,normanniles1/normanniles1_2025-10-23-11-49-45.mp4,99,1,0,101,0
2,normanniles1/normanniles1_2025-10-26-17-59-45.mp4,61,0,0,75,0
3,normanniles2/normanniles2_2025-10-25-16-46-45.mp4,79,1,0,92,0
4,normanniles2/normanniles2_2025-10-25-17-33-45.mp4,78,0,0,88,0
5,normanniles2/normanniles2_2025-10-26-13-10-45.mp4,61,0,0,53,2
6,normanniles3/normanniles3_2025-10-20-06-00-45.mp4,3,0,0,4,0
7,normanniles3/normanniles3_2025-10-23-11-49-45.mp4,79,0,0,118,0
8,normanniles3/normanniles3_2025-10-26-17-59-45.mp4,69,0,0,109,1
9,normanniles4/normanniles4_2025-10-20-06-00-45.mp4,2,0,0,3,0


Open each annotated.mp4 under: /content/drive/MyDrive/CP-1 Dataset/Junction_v4/yolo11/junction_01/pilots/2d3c39a6b187/clips
Rough full-run estimate from this pilot: 775.5 GPU/runtime hours; I/O and scene load can change it.


HTML(value='Open the annotated pilot videos in Drive and inspect each gate, arrow and route. This button recor…

Button(button_style='success', description='I reviewed pilot gates and directions', layout=Layout(width='350px…

Output()

In [6]:
# Cell 7 — full inventory, no sampling. Completed clips resume; interrupted clips restart at zero.
if STAGE=='batch':
    calibration_tools.require_pilot_reviews(CALIBRATION_FOLDER,CALIBRATION,DETECTOR,SETTINGS)
    FEATURES,PENDING=junction.run_batch(DRIVE_BASE,RUN_FOLDER,VIDEOS,CALIBRATION,DETECTOR,SETTINGS)
    print('Completed:',len(FEATURES),'Pending/failed:',len(PENDING))
    if PENDING:
        print('Inspect processing_failures.csv. No failed or partial clip is treated as a completed count.')
else:
    print('All-video processing runs only with STAGE="batch".')

All-video processing runs only with STAGE="batch".


In [7]:
# Cell 8 — labelled next-segment forecasting, after all planned clips complete.
if STAGE=='evaluate':
    contract_path=RUN_FOLDER/'contract.json'
    if not contract_path.is_file():raise FileNotFoundError('Run the batch stage first.')
    contract=json.loads(contract_path.read_text())
    # Recheck current configuration/inventory before trusting the completed index.
    signature=junction.freeze_run(RUN_FOLDER,VIDEOS,CALIBRATION,DETECTOR,SETTINGS)
    FEATURES,PENDING=junction.collect_features(RUN_FOLDER,VIDEOS,signature)
    if PENDING:raise RuntimeError(f'{len(PENDING)} videos are incomplete; resolve the saved failures and resume batch first.')
    if not LABELS_PATH.is_file():raise FileNotFoundError('Set LABELS_PATH to the supplied congestion-label CSV.')
    LABELS=pd.read_csv(LABELS_PATH)
    EVALUATION_FOLDER=RUN_FOLDER/'evaluations'/EVALUATION_NAME
    METRICS=forecasting.evaluate(FEATURES,LABELS,EVALUATION_FOLDER,TEST_START,TEST_END_EXCLUSIVE,HISTORY_CLIPS)
    display(METRICS)
    display(pd.read_csv(EVALUATION_FOLDER/'split_class_distribution.csv'))
    import matplotlib.pyplot as plt
    matrix=pd.read_csv(EVALUATION_FOLDER/'confusion_matrix.csv',index_col=0)
    fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(matrix.values,cmap='Blues')
    for i in range(4):
        for j in range(4):
            r,g,b,_=im.cmap(im.norm(matrix.iloc[i,j]));colour='black' if .2126*r+.7152*g+.0722*b>.5 else 'white'
            ax.text(j,i,str(matrix.iloc[i,j]),ha='center',va='center',color=colour,fontsize=14)
    ax.set_xticks(range(4),forecasting.CLASSES,rotation=25,ha='right');ax.set_yticks(range(4),forecasting.CLASSES)
    ax.set_xlabel('Predicted future class');ax.set_ylabel('Supplied future class')
    fig.tight_layout();fig.savefig(EVALUATION_FOLDER/'confusion_matrix.png',dpi=250);plt.show()
    print('Default dates were previously examined. No independent confirmation or counting accuracy is implied.')
else:
    print('Forecasting runs only with STAGE="evaluate". Unlabelled videos contribute counts/history but cannot be scored targets.')

Forecasting runs only with STAGE="evaluate". Unlabelled videos contribute counts/history but cannot be scored targets.


In [8]:
# Cell 9 — save a compact results ZIP to Drive; large per-frame logs stay in their clip folders.
if STAGE=='export' and (RUN_FOLDER/'contract.json').is_file():
    signature=junction.freeze_run(RUN_FOLDER,VIDEOS,CALIBRATION,DETECTOR,SETTINGS)
    FEATURES,PENDING=junction.collect_features(RUN_FOLDER,VIDEOS,signature)
    print('Export coverage — completed:',len(FEATURES),'pending:',len(PENDING))
if STAGE in {'batch','evaluate','export'} and (RUN_FOLDER/'junction_features.csv').is_file():
    archive=calibration_tools.export_bundle(RUN_FOLDER,CALIBRATION_FOLDER,PROGRAM_FOLDER)
    print('Results archive:',archive)
    print('Download it using Google Drive in your browser; this works with VS Code Colab.')
else:
    print('No full-run results archive yet. Pilot outputs are already saved in Drive.')

No full-run results archive yet. Pilot outputs are already saved in Drive.


### Manual counting evaluation (after gate calibration)

The pilot creates `manual_journey_counts.csv` and `manual_gate_counts.csv`. Keep these separate from old single-tripwire annotations. For journeys, count a physical vehicle only when both its entry and departure gates are visible within the clip; record incomplete/ambiguous journeys separately. For gate counts, record each entry/exit event by class. Enter zeros explicitly and set every scored row's `annotation_status` to `complete` only after reviewing the full clip. Templates initially contain blanks, not invented zeros.

After completing a copy of a template, run in an extra cell:

```python
report=forecasting.evaluate_manual_counts(
    PILOT_FEATURES,CALIBRATION,
    PILOT_FOLDER/'manual_annotations'/'manual_journey_counts.csv',
    PILOT_FOLDER/'manual_evaluation',kind='journeys')
print(report)
```

Use `kind='gates'` with `manual_gate_counts.csv` for gate-event errors. This measures counting on the annotated clips only; a pilot selected during development is not an independent final counting test. Freeze gates, features and validation choices before testing on new labelled recording dates. For a detector comparison, repeat with `DETECTOR='yolov8'`, the same gates, inventory, history and test protocol. Its pilot review and outputs are separate.